# MMDL · E5 데이터 점검 / E6 중복 검사

**사용법: 원본을 Google Drive에 올리고, 3번 설정 셀의 경로를 맞춘 뒤 위에서부터 실행하세요.**

이 노트북은 제공된 `train_messages.jsonl`과 `reservoir_candidates.jsonl`을 문항 ID로 연결합니다. 설명과 보고서는 한국어이며 원본 문제·풀이는 번역하지 않습니다.

| 단계 | 처리 내용 | 자원 |
|---|---|---|
| E5 | 전체 문항·풀이 보유량·정답 필드 일치·상태 충돌·출처·이미지 해시/손상/연결 검사 | CPU / 디스크 |
| E6 | 학습 묶음 안/사이 텍스트·이미지 중복 후보, MMMU 전체 split·MMMU-Pro와 직접 비교 | CPU + 선택적 GPU |
| CLIP | 평가셋 이미지와 시각적 유사도 비교 | A100 사용 |

- Qwen을 로드하거나 학습하지 않습니다. 공식 MMMU 채점기·평가 프롬프트도 변경하지 않습니다.
- 원본을 수정·삭제하지 않습니다. 유사도 후보는 사람이 검토하며 자동으로 정답을 수정하지 않습니다.
- 기존 제공자의 중복 검사 기록과 **이번 실행의 직접 검사 결과를 분리**합니다.
- MMMU revision은 프로젝트 고정값을 사용합니다. MMMU-Pro는 첫 실행 시 `main`을 정확한 커밋으로 고정하고 재개 시 재사용합니다.
- **E6는 텍스트 정확 일치, MinHash 후보+Jaccard, 이미지 SHA256·픽셀 해시·pHash, 선택적 CLIP을 수행합니다. OCR 재실행·번역 문항 의미 검색은 포함하지 않습니다. 미검출은 무오염 보증이 아닙니다.**
- E8 풀이 내용 검증과 학습용 변환은 수행하지 않습니다. `E8_검토후보_ID.jsonl`도 학습 승인 목록이 아닙니다.
- 코랩의 **런타임 → 런타임 유형 변경 → A100 GPU**를 선택하세요. E5에서 GPU 이용률이 낮은 것은 정상입니다.


## 1. 필요한 패키지 설치

새 데이터 점검 런타임에서 실행하세요. 이 노트북의 CLIP용 `transformers` 버전은 Stage 0의 Qwen 평가 환경과 별개입니다. PyTorch/CUDA는 코랩에 설치된 것을 사용합니다.


In [ ]:
#@title 1-1. 점검 패키지 설치
import os, sys, subprocess
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
packages = [
    "zstandard==0.25.0", "ImageHash==4.3.2", "datasketch==2.0.0",
    "pyarrow==26.0.1", "huggingface_hub==0.36.0", "transformers==4.57.2",
    "Pillow>=10,<14", "scipy>=1.10", "tqdm>=4.66"
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("설치 완료. 이후 셀을 계속 실행하세요.")


설치 완료. 이후 셀을 계속 실행하세요.


In [ ]:
#@title 1-2. Google Drive 연결 및 GPU 확인
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import shutil
print("/content 여유 공간:", round(shutil.disk_usage("/content").free / 1024**3, 1), "GiB")
def gpu_description():
    if not shutil.which("nvidia-smi"):
        return "GPU 정보 없음"
    result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
    return result.stdout.strip() or "GPU 정보 없음"
print(gpu_description())
print("E5는 CPU로 실행합니다. 기본 설정의 CLIP 단계에는 GPU가 필요합니다.")


Mounted at /content/drive
/content 여유 공간: 235.2 GiB
NVIDIA L4, 23034 MiB
E5는 CPU로 실행합니다. 기본 설정의 CLIP 단계에는 GPU가 필요합니다.


## 2. 분석 코드 준비

다음 셀은 노트북에 포함된 코드를 `/content`에 저장합니다. 별도 코드 파일을 업로드할 필요가 없습니다. 코드는 셀을 펼쳐 확인할 수 있습니다.


In [ ]:
%%writefile /content/mmdl_e56_engine.py
"""MMDL E5/E6 v1.0 — read-only audit; no SFT or answer generation.

Reports are Korean. Original multilingual questions/solutions remain unchanged.
New detections and source-provided evidence are stored separately.
"""
import ast
import csv
import hashlib
import io
import json
import os
from pathlib import Path, PurePosixPath
import re
import shutil
import sqlite3
import tarfile
import time
import unicodedata
import zipfile
from collections import Counter, defaultdict
from contextlib import contextmanager

VERSION = '1.0.0'
MIB = 1024**2
IMAGE_EXT = {'.jpg', '.jpeg', '.png', '.webp', '.gif', '.bmp', '.tif', '.tiff'}
SUBJECTS = ['Accounting','Agriculture','Architecture_and_Engineering','Art','Art_Theory',
 'Basic_Medical_Science','Biology','Chemistry','Clinical_Medicine','Computer_Science','Design',
 'Diagnostics_and_Laboratory_Medicine','Economics','Electronics','Energy_and_Power','Finance',
 'Geography','History','Literature','Manage','Marketing','Materials','Math','Mechanical_Engineering',
 'Music','Pharmacy','Physics','Psychology','Public_Health','Sociology']
BENCHMARKS = [
 {'repo':'MMMU/MMMU', 'revision':'98e6ac0cb9b7b2cd2c991b85a50762edc4aedc68'},
 {'repo':'MMMU/MMMU_Pro', 'revision':'main'}]
DEFAULTS = {'seed':42, 'phash_distance':6, 'text_jaccard':0.85, 'lsh_threshold':0.60,
 'num_perm':128, 'min_text_chars':30, 'max_lsh_candidates':500,
 'clip_enabled':True, 'clip_model':'openai/clip-vit-base-patch32',
 'clip_revision':'3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268',
 'clip_threshold':0.95, 'clip_top_k':5, 'clip_batch_size':64,
 'checkpoint_every':500, 'max_line_bytes':32*MIB, 'max_image_pixels':50000000,
 'max_extract_gb':40, 'expected_pro_rows_per_config':1730, 'benchmarks':BENCHMARKS}


def js(o):
    return json.dumps(o, ensure_ascii=False, sort_keys=True, default=str)


def write_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
    os.replace(tmp, path)


def sha_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda:f.read(4*MIB), b''): h.update(chunk)
    return h.hexdigest()


def normalize(text):
    # Do not remove operators, digits or punctuation: 1.5 and 15 must stay distinct.
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', str(text or ''))).strip().casefold()


def parse_options(value):
    if value is None or value == '': return []
    if isinstance(value, str):
        try: value = json.loads(value)
        except ValueError:
            try: value = ast.literal_eval(value)
            except (ValueError, SyntaxError): raise ValueError('선택지를 목록으로 해석할 수 없음')
    if not isinstance(value, list): raise ValueError('선택지 자료형이 목록이 아님')
    return [str(x) for x in value]


def text_of(content):
    if isinstance(content, str): return content
    if isinstance(content, list):
        return '\n'.join(str(x.get('text','')) for x in content if isinstance(x,dict) and x.get('type')=='text')
    return ''


def answer_norm(value):
    s = str(value).strip()
    m = re.fullmatch(r'(?:Answer:\s*)?\(?([A-Z])\)?[.]?', s, re.I)
    return m.group(1).upper() if m else normalize(s)


def script_hint(text):
    counts = {'한글':len(re.findall(r'[가-힣]',text)), '한자':len(re.findall(r'[\u4e00-\u9fff]',text)),
              '라틴문자':len(re.findall(r'[A-Za-z]',text))}
    return max(counts,key=counts.get) if sum(counts.values()) else '기타'


def rel_path(value):
    s = str(value).replace('\\','/')
    while s.startswith('./'): s=s[2:]
    p=PurePosixPath(s)
    if not s or p.is_absolute() or '..' in p.parts or re.match(r'^[A-Za-z]:',s) or '://' in s:
        raise ValueError('안전한 상대 이미지 경로가 아님')
    return p.as_posix()


def chargrams(text):
    text=normalize(text)
    return {text[i:i+5] for i in range(max(0,len(text)-4))}


def image_stats(path, max_pixels=50000000):
    from PIL import Image, ImageOps
    import imagehash
    digest=sha_file(path)
    with Image.open(path) as im:
        if im.width*im.height>max_pixels: raise ValueError('이미지 픽셀 상한 초과: 별도 검토 필요')
        frames=getattr(im,'n_frames',1)
        im.verify()
    with Image.open(path) as im:
        im.seek(0); im.load()
        rgb=ImageOps.exif_transpose(im).convert('RGBA')
        bg=Image.new('RGBA',rgb.size,'white'); bg.alpha_composite(rgb)
        rgb=bg.convert('RGB')
        pix=hashlib.sha256(str(rgb.size).encode()+rgb.tobytes()).hexdigest()
        return {'sha256':digest, 'pixel_sha256':pix, 'phash':str(imagehash.phash(rgb,hash_size=8)),
                'width':rgb.width,'height':rgb.height,'frames':frames}


@contextmanager
def tar_reader(path):
    if str(path).lower().endswith(('.zst','.zstd')):
        import zstandard
        with open(path,'rb') as f, zstandard.ZstdDecompressor().stream_reader(f) as r:
            with tarfile.open(fileobj=r,mode='r|') as t: yield t
    else:
        with tarfile.open(path,mode='r|*') as t: yield t


def unpack_safe(source, dest, max_gb=40):
    """No links/devices, traversal, duplicate member paths, or silent reuse of partial output."""
    dest=Path(dest); dest.mkdir(parents=True,exist_ok=True)
    total=0; seen=set(); last=time.monotonic()
    def put(name,size,opener):
        nonlocal total,last
        name=rel_path(name)
        if name in seen: raise ValueError('압축 내부 경로 중복: '+name)
        seen.add(name); total+=size
        if total>max_gb*1024**3: raise ValueError('압축 해제 크기 상한 초과')
        target=dest/name; target.parent.mkdir(parents=True,exist_ok=True)
        with opener() as src, target.open('wb') as dst: shutil.copyfileobj(src,dst,4*MIB)
        if target.stat().st_size!=size: raise ValueError('압축파일 크기 불일치: '+name)
        if time.monotonic()-last>10:
            print(f'압축 해제: {len(seen):,}개 / {total/1024**3:.2f} GiB',flush=True); last=time.monotonic()
    if source.suffix.lower()=='.zip':
        with zipfile.ZipFile(source) as z:
            for m in z.infolist():
                if m.is_dir(): continue
                if (m.external_attr>>16)&0o170000==0o120000: raise ValueError('심볼릭 링크는 허용하지 않음')
                put(m.filename,m.file_size,lambda m=m:z.open(m))
    else:
        with tar_reader(source) as t:
            for m in t:
                if m.isdir(): continue
                if not m.isfile(): raise ValueError('일반 파일이 아닌 압축 항목: '+m.name)
                put(m.name,m.size,lambda m=m:t.extractfile(m))
                t.members.clear()
    return {'files':len(seen),'bytes':total}


class AuditProject:
    def __init__(self, inputs, output, work, config=None):
        self.inputs={str(k):str(v) for k,v in inputs.items()}
        if not self.inputs or any(not re.fullmatch('[A-Za-z0-9_-]+',k) for k in self.inputs):
            raise ValueError('묶음 이름은 영문/숫자/밑줄/하이픈을 사용하세요.')
        self.cfg={**DEFAULTS,**(config or {})}
        if not (0<self.cfg['lsh_threshold']<=self.cfg['text_jaccard']<=1): raise ValueError('텍스트 임계값 확인 필요')
        self.output=Path(output); self.work=Path(work)
        self.output.mkdir(parents=True,exist_ok=True); self.work.mkdir(parents=True,exist_ok=True)
        self.roots={}; self.last_backup=time.monotonic()
        code_sha=sha_file(__file__)
        identity={'version':VERSION,'code_sha256':code_sha,'inputs':self.inputs,'config':self.cfg}
        config_path=self.output/'실행설정.json'
        if config_path.exists() and json.loads(config_path.read_text())!=identity:
            raise RuntimeError('입력·설정·코드가 이전 실행과 다릅니다. RUN_NAME을 새 이름으로 바꾸세요.')
        write_json(config_path,identity)
        run_key=hashlib.sha256(js(identity).encode()).hexdigest()[:16]
        self.work=self.work/run_key; self.work.mkdir(exist_ok=True)
        db_path=self.work/'audit.sqlite'
        if not db_path.exists() and (self.output/'audit.sqlite').exists():
            print('Drive 체크포인트 복원 중...',flush=True); shutil.copy2(self.output/'audit.sqlite',db_path)
        self.db=sqlite3.connect(db_path); self.db.row_factory=sqlite3.Row
        self.db.executescript('''
        PRAGMA journal_mode=WAL;
        CREATE TABLE IF NOT EXISTS state(k TEXT PRIMARY KEY, v TEXT);
        CREATE TABLE IF NOT EXISTS meta(pkg TEXT,id TEXT,data TEXT,PRIMARY KEY(pkg,id));
        CREATE TABLE IF NOT EXISTS examples(rid TEXT PRIMARY KEY,pkg TEXT,id TEXT,data TEXT);
        CREATE INDEX IF NOT EXISTS example_id ON examples(id);
        CREATE TABLE IF NOT EXISTS images(ikey TEXT PRIMARY KEY,pkg TEXT,rel TEXT,data TEXT);
        CREATE TABLE IF NOT EXISTS refs(rid TEXT,ikey TEXT,expected TEXT,PRIMARY KEY(rid,ikey));
        CREATE INDEX IF NOT EXISTS refs_image ON refs(ikey);
        CREATE TABLE IF NOT EXISTS issues(stage TEXT,rid TEXT,code TEXT,severity TEXT,detail TEXT,
           UNIQUE(stage,rid,code,detail));
        CREATE INDEX IF NOT EXISTS issue_code_detail ON issues(code,detail);
        CREATE TABLE IF NOT EXISTS bench(bid TEXT PRIMARY KEY,repo TEXT,config TEXT,split TEXT,id TEXT,q TEXT,options TEXT);
        CREATE TABLE IF NOT EXISTS bench_images(ikey TEXT PRIMARY KEY,data TEXT);
        CREATE TABLE IF NOT EXISTS bench_refs(bid TEXT,ikey TEXT,PRIMARY KEY(bid,ikey));
        CREATE TABLE IF NOT EXISTS hits(rid TEXT,match_id TEXT,kind TEXT,score REAL,detail TEXT,
           UNIQUE(rid,match_id,kind,detail));
        CREATE TABLE IF NOT EXISTS groups(gid TEXT PRIMARY KEY,kind TEXT,members TEXT,detail TEXT);
        CREATE TABLE IF NOT EXISTS features(ikey TEXT PRIMARY KEY,vector BLOB);
        '''); self.db.commit()

    def get(self,k,default=None):
        row=self.db.execute('SELECT v FROM state WHERE k=?',(k,)).fetchone()
        return json.loads(row[0]) if row else default

    def set(self,k,v): self.db.execute('INSERT OR REPLACE INTO state VALUES(?,?)',(k,js(v)))

    def issue(self,rid,code,detail='',severity='검토',stage='E5'):
        self.db.execute('INSERT OR IGNORE INTO issues VALUES(?,?,?,?,?)',(stage,rid,code,severity,str(detail)))

    def hit(self,rid,other,kind,score,detail=''):
        self.db.execute('INSERT OR IGNORE INTO hits VALUES(?,?,?,?,?)',(rid,other,kind,float(score),str(detail)))

    def checkpoint(self,force=False):
        self.db.commit()
        if not force and time.monotonic()-self.last_backup<90: return
        local=self.work/'backup.sqlite'
        target=sqlite3.connect(local)
        try:
            self.db.backup(target)
            target.commit()
            target.execute('PRAGMA wal_checkpoint(TRUNCATE)')
        finally: target.close()
        tmp=self.output/'audit.sqlite.pending'; shutil.copyfile(local,tmp)
        os.replace(tmp,self.output/'audit.sqlite'); self.last_backup=time.monotonic()
        print('중간 결과 Drive 저장 완료',flush=True)

    def stage_inputs(self):
        for pkg,name in self.inputs.items():
            src=Path(name)
            if not src.exists(): raise FileNotFoundError(f'입력 경로 없음: {src}\nWindows C: 경로 대신 Drive 경로를 넣으세요.')
            if src.is_dir():
                matches=[p.parent for p in src.rglob('train_messages.jsonl') if (p.parent/'reservoir_candidates.jsonl').is_file()]
                if len(matches)!=1: raise ValueError(f'{pkg}: 학습/메타데이터 쌍을 가진 폴더가 {len(matches)}개입니다: {matches}')
                root=matches[0]
                if src.resolve() in self.output.resolve().parents or src.resolve()==self.output.resolve():
                    raise ValueError('출력 폴더를 입력 폴더 내부에 둘 수 없습니다.')
                fp={n:sha_file(root/n) for n in ['train_messages.jsonl','reservoir_candidates.jsonl']}
                image_manifest=hashlib.sha256()
                for p in sorted((root/'images').rglob('*')):
                    if p.is_symlink(): raise ValueError('입력 이미지에 심볼릭 링크가 있습니다.')
                    if p.is_file():
                        st=p.stat(); image_manifest.update(js([str(p.relative_to(root)),st.st_size,st.st_mtime_ns]).encode())
                fp['image_file_listing']=image_manifest.hexdigest()
                # Directories stay mounted; archives are faster for many small Drive files.
            else:
                print(f'{pkg}: 원본 압축파일 해시 확인 중...',flush=True)
                fp={'archive_sha256':sha_file(src),'bytes':src.stat().st_size}
                dest=self.work/('source_'+pkg)
                marker=dest/'_extraction_complete.json'
                if not marker.exists() or json.loads(marker.read_text())!=fp:
                    if dest.exists(): shutil.rmtree(dest)
                    print(f'{pkg}: /content 작업 폴더로 압축 해제 중...',flush=True)
                    unpack_safe(src,dest,self.cfg['max_extract_gb']); write_json(marker,fp)
                matches=[p.parent for p in dest.rglob('train_messages.jsonl') if (p.parent/'reservoir_candidates.jsonl').is_file()]
                if len(matches)!=1: raise ValueError(f'{pkg}: 학습 파일과 메타데이터 쌍이 {len(matches)}개입니다.')
                root=matches[0]
            old=self.get('fingerprint:'+pkg)
            if old is not None and old!=fp: raise RuntimeError('입력 데이터가 바뀌었습니다. 새 RUN_NAME으로 실행하세요: '+pkg)
            self.set('fingerprint:'+pkg,fp); self.roots[pkg]=root
            print(f'{pkg}: 준비됨 — {root}',flush=True)
        self.checkpoint(True)
        return self.roots

    def records(self,path,pkg,kind):
        with open(path,'rb') as f:
            line=0
            while True:
                raw=f.readline(self.cfg['max_line_bytes']+1)
                if not raw: break
                line+=1
                if len(raw)>self.cfg['max_line_bytes']:
                    while raw and not raw.endswith(b'\n'): raw=f.readline(self.cfg['max_line_bytes']+1)
                    yield line,None,'행 크기 상한 초과'; continue
                if not raw.strip(): continue
                try:
                    obj=json.loads(raw.decode('utf-8-sig'))
                    if not isinstance(obj,dict): raise ValueError('JSON 객체가 아님')
                    yield line,obj,None
                except (ValueError,UnicodeError) as exc: yield line,None,str(exc)

    def e5_import(self):
        if not self.roots: self.stage_inputs()
        for pkg,root in self.roots.items():
            for kind,filename in [('meta','reservoir_candidates.jsonl'),('train','train_messages.jsonl')]:
                key='import:'+pkg+':'+kind
                if self.get(key+':done'): continue
                resume=self.get(key+':line',0)
                print(f'E5 {pkg} {kind}: {resume:,}행 이후부터 처리',flush=True)
                last_line=resume
                for line,r,error in self.records(root/filename,pkg,kind):
                    if line<=resume: continue
                    last_line=line; rid=f'{pkg}:{kind}:{line}'
                    if error:
                        self.issue(rid,'JSON_읽기_오류',error,'차단')
                    elif kind=='meta':
                        ident=r.get('id')
                        if not isinstance(ident,str) or not ident.strip(): self.issue(rid,'메타데이터_ID_누락','','차단')
                        else:
                            fields=['id','question','answer','options','question_type','subject','subfield',
                              'source_family','source_dataset','source_revision','source_split','source_license',
                              'training_use_allowed','reasoning','rationale','solution','reasoning_verification',
                              'training_admitted','difficulty_verified','visual_dependency_verified','decontamination',
                              'image_paths','image_sha256']
                            small={k:r[k] for k in fields if k in r}
                            trial=r.get('pass4')
                            small['pass4_summary']={k:trial.get(k) for k in ['correct_of_four','score_complete','model','model_revision','sampling']} if isinstance(trial,dict) else None
                            cur=self.db.execute('INSERT OR IGNORE INTO meta VALUES(?,?,?)',(pkg,ident,js(small)))
                            if cur.rowcount==0: self.issue(rid,'메타데이터_ID_중복',ident,'차단')
                    else: self._import_train(pkg,line,r)
                    self.set(key+':line',line)
                    if line%self.cfg['checkpoint_every']==0:
                        self.checkpoint(); print(f'  {line:,}행 처리',flush=True)
                self.set(key+':line',last_line); self.set(key+':done',True); self.checkpoint(True)
        self.set('e5_import_done',True); self.checkpoint(True)

    def _import_train(self,pkg,line,r):
        rid=f'{pkg}:train:{line}'; ident=r.get('id')
        if not isinstance(ident,str) or not ident.strip():
            ident=f'__missing_{line}'; self.issue(rid,'학습_ID_누락','','차단')
        row=self.db.execute('SELECT data FROM meta WHERE pkg=? AND id=?',(pkg,ident)).fetchone()
        m=json.loads(row[0]) if row else {}
        if not row: self.issue(rid,'대응_메타데이터_없음',ident,'차단')
        if self.db.execute("SELECT 1 FROM issues WHERE code='메타데이터_ID_중복' AND detail=? AND rid LIKE ? LIMIT 1",(ident,pkg+':meta:%')).fetchone():
            self.issue(rid,'대응_메타데이터_ID_중복',ident,'차단')
        messages=r.get('messages',[])
        if not isinstance(messages,list): messages=[]; self.issue(rid,'메시지_형식_오류','','차단')
        valid=[x for x in messages if isinstance(x,dict)]
        users=[x for x in valid if x.get('role')=='user']; assistants=[x for x in valid if x.get('role')=='assistant']
        if len(users)!=1 or len(assistants)!=1: self.issue(rid,'단일_QA_메시지_아님',f'user={len(users)}, assistant={len(assistants)}','차단')
        user='\n'.join(text_of(x.get('content')) for x in users)
        target=text_of(assistants[-1].get('content')) if assistants else ''
        if not user.strip(): self.issue(rid,'사용자_문제_누락','','차단')
        if not target.strip(): self.issue(rid,'학습_정답_누락','','차단')
        gold=m.get('answer'); question=m.get('question')
        if not isinstance(question,str) or not question.strip():
            question=user; self.issue(rid,'원본_문제_필드_없음','메시지 본문을 검색용으로 사용')
        if gold is None or gold=='': self.issue(rid,'원본_정답_누락','','차단')
        elif answer_norm(gold)!=answer_norm(target): self.issue(rid,'학습_정답과_원본_정답_다름',f'{target!r} / {gold!r}')
        try: options=parse_options(m.get('options',[]))
        except ValueError as exc: options=[]; self.issue(rid,'선택지_형식_오류',str(exc),'차단')
        if options and not re.fullmatch('[A-Z]',str(gold or '').strip()): self.issue(rid,'객관식_정답이_글자_아님',str(gold))
        if options and re.fullmatch('[A-Z]',str(gold or '').strip()) and ord(str(gold).strip())-65>=len(options):
            self.issue(rid,'객관식_정답_범위_오류',str(gold),'차단')
        subject=r.get('subject') or m.get('subject') or '미분류'
        if subject not in SUBJECTS: self.issue(rid,'과목_라벨_확인',str(subject))
        if r.get('subject') and m.get('subject') and r['subject']!=m['subject']: self.issue(rid,'과목_라벨_불일치')
        if m.get('training_use_allowed') is False: self.issue(rid,'학습_이용_불가_기록','','차단')
        elif m.get('training_use_allowed') is not True: self.issue(rid,'학습_이용_근거_확인')
        for flag in ['training_admitted','difficulty_verified']:
            if m.get(flag) is False: self.issue(rid,'상태충돌_'+flag,'학습 파일에 있으나 해당 값은 false')
        for k in ['source_dataset','source_revision','source_license']:
            if not m.get(k): self.issue(rid,'출처정보_누락_'+k)
        solutions={k:m[k] for k in ['reasoning','rationale','solution'] if isinstance(m.get(k),str) and m[k].strip()}
        if len(set(solutions.values()))>1: self.issue(rid,'복수_풀이_내용_다름','자동으로 하나를 정답 풀이로 선택하지 않음')
        paths=r.get('images',[])
        if not isinstance(paths,list): paths=[]; self.issue(rid,'이미지_목록_형식_오류','','차단')
        if not paths: self.issue(rid,'이미지_없는_문항','텍스트 전용 문항 여부 검토')
        placeholders=sum(1 for u in users if isinstance(u.get('content'),list) for v in u['content'] if isinstance(v,dict) and v.get('type')=='image')
        if placeholders!=len(paths): self.issue(rid,'이미지_토큰_개수_불일치',f'{placeholders} / {len(paths)}','차단')
        expected={}
        mp=m.get('image_paths'); mh=m.get('image_sha256')
        if isinstance(mp,list) and isinstance(mh,list):
            for p,h in zip(mp,mh):
                try: expected[rel_path(p)]=str(h)
                except ValueError: pass
        clean=[]
        for p in paths:
            try:
                if not isinstance(p,str): raise ValueError('문자열 경로가 아님')
                p=rel_path(p); clean.append(p); ikey=pkg+'/'+p
                self.db.execute('INSERT OR IGNORE INTO images VALUES(?,?,?,NULL)',(ikey,pkg,p))
                self.db.execute('INSERT OR IGNORE INTO refs VALUES(?,?,?)',(rid,ikey,expected.get(p)))
            except ValueError as exc: self.issue(rid,'이미지_경로_오류',str(exc),'차단')
        if isinstance(mp,list):
            try:
                if [rel_path(x) for x in mp]!=clean: self.issue(rid,'이미지_목록_순서_불일치','학습 파일과 메타데이터 비교','차단')
            except (ValueError,TypeError): self.issue(rid,'메타데이터_이미지_경로_오류','','차단')
        dc=m.get('decontamination')
        if not isinstance(dc,dict): dc={}; self.issue(rid,'기존_중복검사_기록_없음')
        required={'MMMU_dev','MMMU_validation','MMMU_test','MMMU-Pro'}
        against=dc.get('checked_against',[])
        if dc.get('complete') is not True or not isinstance(against,list) or not required.issubset(set(map(str,against))):
            self.issue(rid,'기존_중복검사_범위_불완전')
        if dc.get('exact_match') is True or dc.get('ocr_13gram_match') is True: self.issue(rid,'기존_중복_양성_기록','새 E6 결과와 별도로 보류')
        data={'question':question,'options':options,'answer':gold,'train_answer':target,'images':clean,
          'subject':subject,'subfield':r.get('subfield') or m.get('subfield'),
          'source_family':r.get('source_family') or m.get('source_family') or '미상',
          'source_dataset':m.get('source_dataset'),'source_revision':m.get('source_revision'),
          'source_license':m.get('source_license'),'source_split':m.get('source_split'),
          'script_hint':script_hint(question),'solutions':solutions,'reasoning_verification':m.get('reasoning_verification'),
          'stored_decontamination':dc,'pass4_summary':m.get('pass4_summary'),
          'system_prompt':'\n'.join(text_of(x.get('content')) for x in valid if x.get('role')=='system'),
          'user_text':user,'question_type':'객관식' if options else '주관식'}
        self.db.execute('INSERT OR REPLACE INTO examples VALUES(?,?,?,?)',(rid,pkg,ident,js(data)))

    def e5_images(self):
        if not self.get('e5_import_done'): raise RuntimeError('E5 문항 읽기를 먼저 실행하세요.')
        if not self.roots: self.stage_inputs()
        for pkg,root in self.roots.items():
            for p in (root/'images').rglob('*'):
                if p.is_file() and p.suffix.lower() in IMAGE_EXT:
                    rel=str(p.relative_to(root)).replace('\\','/')
                    self.db.execute('INSERT OR IGNORE INTO images VALUES(?,?,?,NULL)',(pkg+'/'+rel,pkg,rel))
        self.db.commit()
        pending=self.db.execute('SELECT ikey,pkg,rel FROM images WHERE data IS NULL').fetchall()
        print(f'E5 이미지 검사: 남은 {len(pending):,}개',flush=True)
        for n,row in enumerate(pending,1):
            path=self.roots[row['pkg']]/row['rel']
            try:
                if not path.is_file(): raise FileNotFoundError('이미지 파일 없음')
                if path.is_symlink() or self.roots[row['pkg']].resolve() not in path.resolve().parents: raise ValueError('이미지 루트 밖 경로')
                result=image_stats(path,self.cfg['max_image_pixels']); result['status']='정상'
            except Exception as exc: result={'status':'오류','error':f'{type(exc).__name__}: {exc}'}
            self.db.execute('UPDATE images SET data=? WHERE ikey=?',(js(result),row['ikey']))
            for ref in self.db.execute('SELECT rid,expected FROM refs WHERE ikey=?',(row['ikey'],)).fetchall():
                if result['status']!='정상': self.issue(ref['rid'],'이미지_읽기_실패',row['rel']+': '+result['error'],'차단')
                elif ref['expected'] and result['sha256']!=ref['expected']: self.issue(ref['rid'],'이미지_SHA256_불일치',row['rel'],'차단')
                if result.get('frames',1)>1: self.issue(ref['rid'],'다중프레임_이미지','중복검사 픽셀/pHash/CLIP은 첫 프레임만 사용')
            if n%self.cfg['checkpoint_every']==0:
                print(f'  이미지 {n:,}/{len(pending):,}',flush=True); self.checkpoint()
        for row in self.db.execute('SELECT pkg,id,COUNT(*) n FROM examples GROUP BY pkg,id HAVING n>1').fetchall():
            for ex in self.db.execute('SELECT rid FROM examples WHERE pkg=? AND id=?',(row['pkg'],row['id'])).fetchall():
                self.issue(ex['rid'],'묶음내_학습_ID_중복',row['id'],'차단')
        self.set('e5_done',True); self.checkpoint(True)

    def examples(self):
        return [(r['rid'],r['pkg'],r['id'],json.loads(r['data'])) for r in self.db.execute('SELECT * FROM examples ORDER BY rid')]

    def image_path(self,ikey):
        if ikey.startswith('bench:'): return self.work/'benchmark_images'/(ikey[6:]+'.img')
        row=self.db.execute('SELECT pkg,rel FROM images WHERE ikey=?',(ikey,)).fetchone()
        return self.roots[row['pkg']]/row['rel']

    def load_benchmarks(self):
        """Read question/options/images only. Gold answers/explanations are not imported."""
        from huggingface_hub import HfApi, snapshot_download
        import pyarrow.parquet as pq
        self.set('bench_done',False); self.checkpoint(True)
        imdir=self.work/'benchmark_images'; imdir.mkdir(exist_ok=True)
        coverage=[]; problems=[]
        for spec in self.cfg['benchmarks']:
            repo=spec['repo']; lockkey='bench_revision:'+repo
            revision=self.get(lockkey)
            if not revision:
                revision=HfApi().dataset_info(repo,revision=spec['revision']).sha
                self.set(lockkey,revision); self.checkpoint(True)
            print(f'평가 데이터 다운로드: {repo} @ {revision}',flush=True)
            folder=Path(snapshot_download(repo_id=repo,repo_type='dataset',revision=revision,
                      allow_patterns=['*.parquet'],cache_dir=str(self.work/'hf_cache'),max_workers=4))
            files=sorted(folder.rglob('*.parquet'))
            if not files: raise RuntimeError('평가용 parquet 파일이 없습니다: '+repo)
            for file in files:
                relative=file.relative_to(folder).as_posix(); config=str(PurePosixPath(relative).parent)
                split=file.name.split('-')[0]
                pf=pq.ParquetFile(file)
                columns=[x for x in pf.schema_arrow.names if x in {'id','question','options','subject','image'} or re.fullmatch(r'image_\d+',x)]
                if 'id' not in columns: raise RuntimeError('평가셋 id 열이 없습니다: '+relative)
                count=0
                for batch in pf.iter_batches(batch_size=8,columns=columns):
                    for r in batch.to_pylist():
                        count+=1; ident=str(r['id']); bid=f'{repo}|{relative}|{ident}'
                        try: opts=parse_options(r.get('options'))
                        except ValueError as exc:
                            opts=[]; problems.append(bid+': '+str(exc)); self.issue(bid,'벤치마크_선택지_오류',str(exc),'차단','E6')
                        self.db.execute('INSERT OR IGNORE INTO bench VALUES(?,?,?,?,?,?,?)',
                            (bid,repo,config,split,ident,str(r.get('question') or ''),js(opts)))
                        nimages=0
                        for key in columns:
                            if not (key=='image' or re.fullmatch(r'image_\d+',key)) or r.get(key) is None: continue
                            payload=r[key]
                            if isinstance(payload,dict):
                                raw=payload.get('bytes')
                                if raw is None and payload.get('path'):
                                    candidate=folder/rel_path(payload['path'])
                                    raw=candidate.read_bytes()
                            elif isinstance(payload,bytes): raw=payload
                            else: raw=None
                            if not raw:
                                problems.append(bid+': 이미지 바이트 없음'); self.issue(bid,'벤치마크_이미지_없음',key,'차단','E6'); continue
                            nimages+=1; sha=hashlib.sha256(raw).hexdigest(); ikey='bench:'+sha
                            path=imdir/(sha+'.img')
                            if not path.exists(): path.write_bytes(raw)
                            if not self.db.execute('SELECT 1 FROM bench_images WHERE ikey=?',(ikey,)).fetchone():
                                try: data=image_stats(path,self.cfg['max_image_pixels']); data['status']='정상'
                                except Exception as exc:
                                    data={'status':'오류','error':str(exc)}; problems.append(bid+': '+str(exc))
                                    self.issue(bid,'벤치마크_이미지_오류',str(exc),'차단','E6')
                                self.db.execute('INSERT INTO bench_images VALUES(?,?)',(ikey,js(data)))
                            self.db.execute('INSERT OR IGNORE INTO bench_refs VALUES(?,?)',(bid,ikey))
                        if not nimages:
                            problems.append(bid+': 이미지 수 0'); self.issue(bid,'벤치마크_이미지_수_0','','차단','E6')
                        if count%self.cfg['checkpoint_every']==0: self.checkpoint()
                coverage.append({'repository':repo,'revision':revision,'file':relative,'rows':count,
                                 'parquet_rows':pf.metadata.num_rows,'split':split,'config':config})
                print(f'  {relative}: {count:,}문항',flush=True)
                self.checkpoint()
        # Check completeness independently of source-provided decontamination flags.
        mcounts={r['split']:r['n'] for r in self.db.execute("SELECT split,COUNT(*) n FROM bench WHERE repo='MMMU/MMMU' GROUP BY split")}
        if mcounts!={'dev':150,'validation':900,'test':10500}: problems.append('MMMU split별 문항 수 확인 필요: '+js(mcounts))
        pcounts={r['config']:r['n'] for r in self.db.execute("SELECT config,COUNT(*) n FROM bench WHERE repo='MMMU/MMMU_Pro' GROUP BY config")}
        for required in ['standard (4 options)','standard (10 options)','vision']:
            matching=[n for k,n in pcounts.items() if required==k or k.endswith('/'+required)]
            if not matching: problems.append('MMMU-Pro 구성 누락: '+required)
            elif sum(matching)!=self.cfg['expected_pro_rows_per_config']:
                problems.append(f'MMMU-Pro {required} 문항 수 확인 필요: {sum(matching)}')
        bad_images=sum(json.loads(r[0]).get('status')!='정상' for r in self.db.execute('SELECT data FROM bench_images'))
        if bad_images: problems.append(f'평가 이미지 오류 {bad_images}개')
        self.set('bench_coverage',coverage); self.set('bench_problems',problems)
        self.set('bench_done',not problems); self.checkpoint(True)
        write_json(self.output/'벤치마크_범위.json',{'files':coverage,'problems':problems,'complete':not problems})
        if problems: raise RuntimeError('평가셋 준비가 불완전합니다. 벤치마크_범위.json 확인: '+str(problems[:3]))

    def e6_text(self):
        from datasketch import MinHash, MinHashLSH
        if not self.get('e5_done') or not self.get('bench_done'): raise RuntimeError('E5와 평가셋 준비를 먼저 끝내세요.')
        if self.get('text_done'): print('E6 텍스트 검사가 이미 완료되었습니다.'); return
        bench=list(self.db.execute('SELECT bid,q,options FROM bench ORDER BY bid'))
        ex=self.examples()
        exact=defaultdict(list); option_exact=defaultdict(list); bgrams={}
        lsh=MinHashLSH(threshold=self.cfg['lsh_threshold'],num_perm=self.cfg['num_perm'])
        def signature(grams):
            h=MinHash(num_perm=self.cfg['num_perm'],seed=self.cfg['seed'])
            h.update_batch([g.encode('utf-8') for g in sorted(grams)])
            return h
        for b in bench:
            q=normalize(b['q'])
            if not q: continue  # Vision-only has no textual question; image checks handle it.
            exact[q].append(b['bid'])
            optkey=js([q,sorted(normalize(x) for x in json.loads(b['options']))])
            option_exact[optkey].append(b['bid'])
            if len(q)>=self.cfg['min_text_chars']:
                grams=chargrams(q); bgrams[b['bid']]=grams; lsh.insert(b['bid'],signature(grams))
        internal=MinHashLSH(threshold=self.cfg['lsh_threshold'],num_perm=self.cfg['num_perm'])
        igrams={}; qgroups=defaultdict(list); qogroups=defaultdict(list)
        for n,(rid,pkg,ident,d) in enumerate(ex,1):
            q=normalize(d['question']); opts=sorted(normalize(x) for x in d['options']); qokey=js([q,opts])
            if q:
                qgroups[q].append(rid); qogroups[qokey].append(rid)
                for bid in exact.get(q,[]): self.hit(rid,bid,'평가셋_문제문_정규화_일치',1)
                for bid in option_exact.get(qokey,[]): self.hit(rid,bid,'평가셋_문제와선택지_일치',1)
            if len(q)>=self.cfg['min_text_chars']:
                grams=chargrams(q); mh=signature(grams)
                for index,bank,kind in [(lsh,bgrams,'평가셋_텍스트_유사'),(internal,igrams,'학습간_텍스트_유사')]:
                    candidates=sorted(index.query(mh))
                    if len(candidates)>self.cfg['max_lsh_candidates']:
                        self.issue(rid,'텍스트_후보_상한_도달',f'{kind}: {len(candidates)}개 중 일부 비교','검토','E6')
                    for other in candidates[:self.cfg['max_lsh_candidates']]:
                        g=bank[other]; similarity=len(grams&g)/len(grams|g)
                        if similarity>=self.cfg['text_jaccard']: self.hit(rid,other,kind,similarity)
                internal.insert(rid,mh); igrams[rid]=grams
            if n%self.cfg['checkpoint_every']==0:
                print(f'E6 텍스트 {n:,}/{len(ex):,}',flush=True); self.checkpoint()
        # Groups avoid quadratic pair output for identical common questions.
        for kind,groups in [('학습간_문제문_일치',qgroups),('학습간_문제와선택지_일치',qogroups)]:
            for key,members in groups.items():
                if len(members)<2: continue
                gid=kind+':'+hashlib.sha256(key.encode()).hexdigest()
                self.db.execute('INSERT OR REPLACE INTO groups VALUES(?,?,?,?)',(gid,kind,js(members),'텍스트 일치만으로 동일 이미지/동일 문항을 확정하지 않음'))
        self.set('text_done',True); self.checkpoint(True)

    def e6_images(self):
        import numpy as np
        if not self.get('e5_done') or not self.get('bench_done'): raise RuntimeError('E5와 평가셋 준비를 먼저 끝내세요.')
        if self.get('image_match_done'): print('E6 이미지 해시 검사가 이미 완료되었습니다.'); return
        train=[(r['ikey'],json.loads(r['data'])) for r in self.db.execute('SELECT ikey,data FROM images') if r['data']]
        bench=[(r['ikey'],json.loads(r['data'])) for r in self.db.execute('SELECT * FROM bench_images')]
        train=[x for x in train if x[1].get('status')=='정상']; bench=[x for x in bench if x[1].get('status')=='정상']
        refs=defaultdict(list); bref=defaultdict(list)
        for r in self.db.execute('SELECT * FROM refs'): refs[r['ikey']].append(r['rid'])
        for r in self.db.execute('SELECT * FROM bench_refs'): bref[r['ikey']].append(r['bid'])
        indices={k:defaultdict(list) for k in ['sha256','pixel_sha256']}
        for ikey,data in bench:
            for key in indices: indices[key][data[key]].append(ikey)
        grouped={k:defaultdict(list) for k in indices}
        # Complete Hamming search against benchmark images, using chunked uint64 XOR.
        bkeys=[x[0] for x in bench]; phashes=np.array([int(x[1]['phash'],16) for x in bench],dtype=np.uint64)
        popcount=np.array([bin(i).count('1') for i in range(256)],dtype=np.uint8)
        # 7 disjoint bands guarantee a candidate for distance <=6 (pigeonhole principle).
        radius=self.cfg['phash_distance']
        if not 0<=radius<=12: raise ValueError('pHash 거리는 0~12로 설정하세요.')
        widths=[64//(radius+1)+(i<64%(radius+1)) for i in range(radius+1)]
        bands=[defaultdict(set) for _ in widths]; prior_hash={}
        def band_values(value):
            shift=0
            for w in widths:
                yield (value>>shift)&((1<<w)-1); shift+=w
        for n,(ikey,data) in enumerate(train,1):
            related=refs[ikey]
            for field,index in indices.items():
                grouped[field][data[field]].append(ikey)
                for other in index.get(data[field],[]):
                    for rid in related:
                        for bid in bref[other]: self.hit(rid,bid,'평가셋_이미지_'+field,1,js({'image':ikey,'benchmark_image':other}))
            value=int(data['phash'],16)
            if len(phashes) and related:
                xor=np.bitwise_xor(phashes,np.uint64(value))
                distances=popcount[xor.view(np.uint8).reshape(-1,8)].sum(axis=1)
                for index in np.flatnonzero(distances<=radius):
                    other=bkeys[int(index)]
                    for rid in related:
                        for bid in bref[other]: self.hit(rid,bid,'평가셋_이미지_pHash',int(distances[index]),js({'image':ikey,'benchmark_image':other,'낮을수록유사':True}))
            matches=set()
            for b,v in zip(bands,band_values(value)): matches.update(b.get(v,()))
            # Identical pixel/raw images already have complete group output below.
            near=[o for o in matches if (value^prior_hash[o]).bit_count()<=radius]
            if len(near)>500:
                self.issue(ikey,'학습간_이미지유사_후보_상한',f'{len(near)}개 중 500개; 동일 이미지 그룹은 별도 전수 기록','검토','E6')
            for other in sorted(near)[:500]:
                if not refs[other] or not related: continue
                dist=(value^prior_hash[other]).bit_count()
                for rid in related:
                    for other_rid in refs[other]:
                        if rid!=other_rid: self.hit(rid,other_rid,'학습간_이미지_pHash',dist,js({'image':ikey,'other_image':other}))
            prior_hash[ikey]=value
            for b,v in zip(bands,band_values(value)): b[v].add(ikey)
            if n%self.cfg['checkpoint_every']==0:
                print(f'E6 이미지 {n:,}/{len(train):,}',flush=True); self.checkpoint()
        for field,groups in grouped.items():
            for digest,keys in groups.items():
                members=sorted({rid for key in keys for rid in refs[key]})
                if len(members)>1:
                    self.db.execute('INSERT OR REPLACE INTO groups VALUES(?,?,?,?)',
                      ('image_'+field+':'+digest,'학습간_이미지_'+field,js(members),js({'image_keys':keys,'주의':'같은 이미지의 서로 다른 문제일 수 있음'})))
        self.set('image_match_done',True); self.checkpoint(True)

    def e6_clip(self):
        if not self.cfg['clip_enabled']:
            self.set('clip_status','설정에 의해 미실행'); self.checkpoint(True); return
        if not self.get('bench_done') or not self.get('e5_done'): raise RuntimeError('E5와 평가셋 준비가 필요합니다.')
        if self.get('clip_done'): print('E6 CLIP 검사가 이미 완료되었습니다.'); return
        import numpy as np
        import torch
        from PIL import Image, ImageOps
        from transformers import CLIPModel, AutoProcessor
        if not torch.cuda.is_available(): raise RuntimeError('GPU가 없습니다. GPU 런타임으로 바꾸거나 CLIP_ENABLED=False로 설정하세요.')
        device='cuda'; torch.manual_seed(self.cfg['seed'])
        model=CLIPModel.from_pretrained(self.cfg['clip_model'],revision=self.cfg['clip_revision']).eval().to(device)
        processor=AutoProcessor.from_pretrained(self.cfg['clip_model'],revision=self.cfg['clip_revision'])
        keys=[r['ikey'] for r in self.db.execute('SELECT * FROM images') if r['data'] and json.loads(r['data']).get('status')=='정상']
        bkeys=[r['ikey'] for r in self.db.execute('SELECT * FROM bench_images') if json.loads(r['data']).get('status')=='정상']
        have={r[0] for r in self.db.execute('SELECT ikey FROM features')}
        pending=[k for k in bkeys+keys if k not in have]
        print(f'CLIP 이미지 특징 계산: 남은 {len(pending):,}개',flush=True)
        size=self.cfg['clip_batch_size']
        for start in range(0,len(pending),size):
            current=pending[start:start+size]; images=[]
            for key in current:
                with Image.open(self.image_path(key)) as im:
                    rgba=ImageOps.exif_transpose(im).convert('RGBA')
                    bg=Image.new('RGBA',rgba.size,'white'); bg.alpha_composite(rgba); images.append(bg.convert('RGB'))
            inputs=processor(images=images,return_tensors='pt').to(device)
            with torch.inference_mode():
                # Explicit projection works with the pinned transformers 4.57.2 API.
                pooled=model.vision_model(pixel_values=inputs['pixel_values']).pooler_output
                features=model.visual_projection(pooled).float()
                features=features/features.norm(dim=-1,keepdim=True).clamp_min(1e-12)
            vectors=features.cpu().numpy().astype('float32')
            for key,vec in zip(current,vectors): self.db.execute('INSERT OR REPLACE INTO features VALUES(?,?)',(key,vec.tobytes()))
            for im in images: im.close()
            print(f'  CLIP {min(start+size,len(pending)):,}/{len(pending):,}',flush=True); self.checkpoint()
        del model; torch.cuda.empty_cache()
        def vectors_for(ks):
            return np.stack([np.frombuffer(self.db.execute('SELECT vector FROM features WHERE ikey=?',(k,)).fetchone()[0],dtype=np.float32) for k in ks])
        if not bkeys: raise RuntimeError('평가 이미지 특징이 없습니다.')
        reference=torch.from_numpy(vectors_for(bkeys)).to(device)
        refs=defaultdict(list); bref=defaultdict(list)
        for r in self.db.execute('SELECT * FROM refs'): refs[r['ikey']].append(r['rid'])
        for r in self.db.execute('SELECT * FROM bench_refs'): bref[r['ikey']].append(r['bid'])
        top=min(self.cfg['clip_top_k'],len(bkeys))
        for start in range(0,len(keys),256):
            current=keys[start:start+256]
            query=torch.from_numpy(vectors_for(current)).to(device)
            sims=query@reference.T
            values,indices=torch.topk(sims,top,dim=1)
            for i,key in enumerate(current):
                count=int((sims[i]>=self.cfg['clip_threshold']).sum().item())
                if count>top:
                    for rid in refs[key]: self.issue(rid,'CLIP_후보_상한',f'{count}개 중 상위 {top}개만 저장','검토','E6')
                for score,j in zip(values[i].tolist(),indices[i].tolist()):
                    if score<self.cfg['clip_threshold']: continue
                    other=bkeys[j]
                    for rid in refs[key]:
                        for bid in bref[other]: self.hit(rid,bid,'평가셋_이미지_CLIP',score,js({'image':key,'benchmark_image':other}))
            self.checkpoint()
        self.set('clip_done',True); self.set('clip_status','완료'); self.checkpoint(True)
        del reference; torch.cuda.empty_cache()

    def export_reports(self):
        """Export audit tables and review candidates, NEVER a ready-to-train corpus."""
        self.checkpoint(True)
        out=self.output/'reports'; out.mkdir(exist_ok=True)
        def export_csv(name,header,records):
            path=out/name
            with path.open('w',encoding='utf-8-sig',newline='') as f:
                w=csv.writer(f); w.writerow(header)
                for record in records:
                    # Spreadsheet formula injection guard, preserving raw JSON in SQLite.
                    w.writerow([("'"+str(x)) if isinstance(x,str) and x.startswith(('=','+','-','@')) else x for x in record])
        examples=self.examples(); exmap={x[0]:x for x in examples}
        issue_map=defaultdict(list); hits=defaultdict(set); internal=defaultdict(set); groups=defaultdict(set)
        all_issues=list(self.db.execute('SELECT * FROM issues ORDER BY stage,rid,code'))
        for r in all_issues: issue_map[r['rid']].append(r)
        all_hits=list(self.db.execute('SELECT * FROM hits ORDER BY rid,kind,match_id'))
        for r in all_hits:
            if r['kind'].startswith('평가셋_'): hits[r['rid']].add(r['kind'])
            else:
                internal[r['rid']].add(r['kind']); internal[r['match_id']].add(r['kind'])
        all_groups=list(self.db.execute('SELECT * FROM groups ORDER BY gid'))
        for g in all_groups:
            for rid in json.loads(g['members']): groups[rid].add(g['kind'])
        core=bool(self.get('bench_done') and self.get('text_done') and self.get('image_match_done'))
        complete=core and (not self.cfg['clip_enabled'] or bool(self.get('clip_done')))
        issues_count=Counter(r['code'] for r in all_issues)
        stats=Counter(); by_subject=Counter(); by_source=Counter(); by_script=Counter(); by_pkg=Counter(); solution_source=Counter()
        rows=[]; queue=[]; next_e8=[]; evidence=[]
        image_ref_counts=Counter(r[0] for r in self.db.execute('SELECT ikey FROM refs'))
        for rid,pkg,ident,d in examples:
            e5=[x for x in issue_map[rid] if x['stage']=='E5']
            e5_status='차단' if any(x['severity']=='차단' for x in e5) else ('검토' if e5 else '기계검사_이상없음')
            if hits[rid]: e6_status='평가셋_중복의심_검토'
            elif not complete: e6_status='검사_미완료'
            elif any(x['stage']=='E6' for x in issue_map[rid]): e6_status='검사제한_검토'
            else: e6_status='구현범위에서_미검출'
            has_solution=bool(d['solutions']); stats['풀이_있음' if has_solution else '풀이_없음']+=1
            stats['E5_'+e5_status]+=1; stats['E6_'+e6_status]+=1
            by_pkg[pkg]+=1; by_subject[(pkg,d['subject'])]+=1; by_source[(pkg,d['source_family'])]+=1; by_script[(pkg,d['script_hint'])]+=1
            if has_solution: solution_source[(pkg,d['source_family'])]+=1
            kinds=sorted(internal[rid]|groups[rid])
            rows.append([pkg,ident,rid,d['subject'],d['subfield'],d['source_family'],d['question_type'],d['script_hint'],
                         has_solution,';'.join(d['solutions']),max([len(s) for s in d['solutions'].values()] or [0]),
                         len(d['images']),e5_status,e6_status,';'.join(sorted(hits[rid])),';'.join(kinds),
                         ';'.join(x['code'] for x in issue_map[rid]),d['source_license'],d['source_revision']])
            evidence.append({'package':pkg,'id':ident,'rid':rid,'source_provided_evidence':d['stored_decontamination'],
               'current_run_benchmark_hit_kinds':sorted(hits[rid]),'current_run_status':e6_status,
               'mmmu_overlap':None,'explanation':'중복 확정 여부는 수동 검토 전 미정. 미검출은 무오염 보증이 아님.'})
            if e5_status!='기계검사_이상없음' or e6_status!='구현범위에서_미검출' or kinds:
                queue.append([pkg,ident,rid,e5_status,e6_status,';'.join(kinds),';'.join(x['code'] for x in issue_map[rid]),
                              d['question'],js(d['options']),str(d['answer']),js(d['images'])])
            if e5_status=='기계검사_이상없음' and e6_status=='구현범위에서_미검출' and not kinds and has_solution:
                next_e8.append({'package':pkg,'id':ident,'rid':rid,'subject':d['subject'],'source_family':d['source_family'],
                  'solution_fields':list(d['solutions']),'E8_status':'풀이_내용_검증_필요',
                  'training_ready':False,'note':'E5/E6 후보 목록이며 학습 승인 목록이 아님'})
        export_csv('E5_문항별_결과.csv',['묶음','ID','행키','과목','세부분야','출처','문제유형','문자종류_언어아님',
            '풀이존재','풀이필드','최대풀이문자수_토큰아님','이미지수','E5상태','E6상태','평가셋의심종류','학습간의심종류','확인사항','기록된라이선스','출처버전'],rows)
        export_csv('E5_문제목록.csv',['단계','행키','항목','심각도','설명'],[[r[k] for k in ['stage','rid','code','severity','detail']] for r in all_issues])
        export_csv('E5_과목별.csv',['묶음','과목','문항수'],[(p,s,n) for (p,s),n in sorted(by_subject.items())])
        export_csv('E5_출처별.csv',['묶음','출처','문항수','풀이보유문항수'],[(p,s,n,solution_source[p,s]) for (p,s),n in sorted(by_source.items())])
        export_csv('E5_문자종류별.csv',['묶음','문자종류_언어확정아님','문항수'],[(p,s,n) for (p,s),n in sorted(by_script.items())])
        export_csv('검토필요_문항.csv',['묶음','ID','행키','E5상태','E6상태','학습간의심','확인사항','원문문제','선택지','원본정답','이미지경로'],queue)
        export_csv('E6_중복의심_쌍.csv',['학습행키','비교대상','종류','점수_pHash는작을수록유사','근거'],
                   [[r[k] for k in ['rid','match_id','kind','score','detail']] for r in all_hits])
        export_csv('E6_학습내외_중복그룹.csv',['그룹ID','종류','행키목록','근거'],
                   [[r[k] for k in ['gid','kind','members','detail']] for r in all_groups])
        image_rows=[]
        for r in self.db.execute('SELECT * FROM images ORDER BY ikey'):
            data=json.loads(r['data']) if r['data'] else {}
            image_rows.append([r['pkg'],r['rel'],image_ref_counts[r['ikey']],data.get('status','미검사'),
                               data.get('sha256'),data.get('width'),data.get('height'),data.get('frames'),data.get('error','')])
        export_csv('E5_이미지목록.csv',['묶음','상대경로','참조문항수','상태','직접계산_SHA256','너비','높이','프레임수','오류'],image_rows)
        for name,items in [('E6_기존기록과_이번검사.jsonl',evidence),('E8_검토후보_ID.jsonl',next_e8)]:
            with (out/name).open('w',encoding='utf-8') as f:
                for item in items: f.write(js(item)+'\n')
        summary={'문항수':len(examples),'묶음별문항수':dict(by_pkg),'상태별':dict(stats),'문제유형별건수':dict(issues_count),
            'E5_단계완료':bool(self.get('e5_done')),'E6_설정된검사완료':complete,
            'E6_범위':{'평가셋준비':bool(self.get('bench_done')),'텍스트정확및LSH후보검사':bool(self.get('text_done')),
                      '이미지바이트_픽셀_pHash':bool(self.get('image_match_done')),'CLIP':self.get('clip_status','미실행'),
                      'OCR_재실행':False,'번역된문제_의미검색':False},
            '벤치마크_범위':self.get('bench_coverage',[]),'벤치마크_문제':self.get('bench_problems',[]),
            'E6_의심쌍수':len(all_hits),'학습간그룹수':len(all_groups),'검토필요문항수':len(queue),'E8_후보문항수':len(next_e8),
            '주의':['정답과 풀이의 내용적 정확성은 검사하지 않았습니다. E8은 미실행입니다.',
             '기존 검사 기록은 제공된 주장으로 보관하며 이번 검사의 대체 증거로 쓰지 않습니다.',
             '텍스트/이미지 유사도는 중복 후보입니다. 원본은 수정·삭제하지 않습니다.',
             'LSH는 후보 검색이므로 누락 가능성이 있습니다. 번역·크롭·이미지 속 문자 중복을 모두 보장하지 않습니다.',
             'CLIP은 평가셋 대상만 비교하며 상위 K개 후보를 기록합니다. 같은 이미지의 다른 문제도 후보가 됩니다.',
             '시스템 설명과 보고서는 한국어이며 데이터 원문은 번역하지 않습니다.']}
        write_json(out/'요약.json',summary)
        lines=['# MMDL E5·E6 점검 결과','',f'- 전체 학습 행: {len(examples):,}',
               f"- E5 단계 완료: {summary['E5_단계완료']}",f'- E6 설정된 검사 완료: {complete}',
               f'- 풀이 보유: {stats["풀이_있음"]:,} / 풀이 없음: {stats["풀이_없음"]:,}',
               f'- 검토 필요 문항: {len(queue):,}',f'- 다음 E8 검토 후보: {len(next_e8):,}',
               '', '단계 완료는 데이터 전체 합격을 뜻하지 않습니다. 요약.json과 문항별 상태를 함께 확인하세요.',
               '', '## 묶음별 문항 수','']+[f'- {p}: {n:,}' for p,n in by_pkg.items()]
        lines+=['','## 발견된 확인사항','']+[f'- {k}: {n:,}' for k,n in issues_count.most_common()]
        lines+=['','## 한계','']+['- '+s for s in summary['주의']]
        (out/'결과요약.md').write_text('\n'.join(lines),encoding='utf-8')
        # Include reproducibility metadata; omit input data and large checkpoint DB.
        archive=self.output/'MMDL_E5_E6_결과.zip'
        with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
            for p in sorted(out.glob('*')):
                if p.is_file(): z.write(p,'reports/'+p.name)
            for name in ['실행설정.json','벤치마크_범위.json','실행환경.json']:
                if (self.output/name).exists(): z.write(self.output/name,name)
        print('\n보고서 저장:',archive,flush=True)
        print(f'문항 {len(examples):,}개 / 검토 필요 {len(queue):,}개 / E6 완료 {complete}',flush=True)
        return summary

    def safe_step(self, fn):
        """Persist committed progress on interruption; do not mark failed stages complete."""
        try: return fn()
        finally: self.checkpoint(True)


Writing /content/mmdl_e56_engine.py


## 3. 입력 경로와 실행 이름 설정 — 여기만 먼저 수정하세요

PC의 `C:\Users\우성한\Downloads\...` 경로는 코랩에서 접근할 수 없습니다. **원본 데이터**를 Drive의 `내 드라이브/MMDL_data`에 올린 뒤 아래 경로를 맞추세요. 앞서 만든 작은 미리보기 ZIP은 입력으로 사용할 수 없습니다.

기본 경로:

- 기존 데이터: `/content/drive/MyDrive/MMDL_data/mmmu_train_20261016.tar.zst`
- 추가 데이터 폴더: `/content/drive/MyDrive/MMDL_data/mmmu_expansion_mmk12_20261115`

추가 데이터가 ZIP 또는 tar.zst라면 `EXPANSION_PATH`를 그 파일 경로로 바꾸면 됩니다. 폴더 안에 같은 이름의 폴더가 한 번 더 있어도 학습 파일 쌍을 찾아줍니다. 다수의 작은 파일이 있는 Drive 폴더보다 압축파일 방식이 보통 디스크 입출력에 유리합니다.

**재개:** 같은 파일·설정·`RUN_NAME`으로 위에서부터 실행하면 Drive 체크포인트를 복원합니다. 입력이나 검사 임계값·코드를 바꿀 때는 새 `RUN_NAME`을 쓰세요. 폴더 입력은 검사 중 내용을 바꾸지 마세요.


In [ ]:
#@title 3-1. 경로와 이름 설정
BASE_PATH = "/content/drive/MyDrive/MMDL_data/mmmu_train_20261016"  #@param {type:"string"}
EXPANSION_PATH = "/content/drive/MyDrive/MMDL_data/mmmu_expansion_mmk12_20261115"  #@param {type:"string"}
RUN_NAME = "e5e6_20261116_v1"  #@param {type:"string"}
CLIP_ENABLED = True  #@param {type:"boolean"}

INPUTS = {"base": BASE_PATH, "expansion": EXPANSION_PATH}
# 기존 데이터만 먼저 확인하려면 위 줄 대신 다음 줄을 사용하세요.
# INPUTS = {"base": BASE_PATH}

assert RUN_NAME and "/" not in RUN_NAME and "\\" not in RUN_NAME and RUN_NAME not in {".", ".."}
OUTPUT_DIR = Path("/content/drive/MyDrive/MMDL_audit_results") / RUN_NAME
WORK_DIR = Path("/content/mmdl_e56_work")
CONFIG = {
    "clip_enabled": CLIP_ENABLED,
    "clip_batch_size": 64,
    "clip_threshold": 0.95,   # 후보 검색 기준, 중복 확정 기준이 아닙니다.
    "clip_top_k": 5,
    "phash_distance": 6,
    "text_jaccard": 0.85,
    "lsh_threshold": 0.60,
    "checkpoint_every": 500,
}
for label, path in INPUTS.items():
    print(label, "확인됨" if Path(path).exists() else "경로 없음", path)
missing = [p for p in INPUTS.values() if not Path(p).exists()]
if missing:
    raise FileNotFoundError("Drive에 원본을 올리거나 위 경로를 수정하세요: " + str(missing))
print("결과 저장 폴더:", OUTPUT_DIR)


base 확인됨 /content/drive/MyDrive/MMDL_data/mmmu_train_20261016
expansion 확인됨 /content/drive/MyDrive/MMDL_data/mmmu_expansion_mmk12_20261115
결과 저장 폴더: /content/drive/MyDrive/MMDL_audit_results/e5e6_20261116_v1


In [ ]:
#@title 3-2. 점검 프로젝트 열기 / 기존 실행 복원
import importlib, importlib.util, importlib.metadata, json, platform
spec = importlib.util.spec_from_file_location("mmdl_e56_engine", "/content/mmdl_e56_engine.py")
engine = importlib.util.module_from_spec(spec)
spec.loader.exec_module(engine)
if "audit" in globals():
    audit.checkpoint(True)
    audit.db.close()
audit = engine.AuditProject(INPUTS, OUTPUT_DIR, WORK_DIR, CONFIG)
versions = {}
for package in ["torch", "transformers", "huggingface_hub", "pyarrow", "datasketch", "ImageHash", "Pillow", "numpy", "zstandard"]:
    try: versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: versions[package] = "없음"
environment = {"python": sys.version, "platform": platform.platform(), "packages": versions,
               "gpu": gpu_description()}
engine.write_json(OUTPUT_DIR / "실행환경.json", environment)
shutil.copyfile("/content/mmdl_e56_engine.py", OUTPUT_DIR / "mmdl_audit.py")
audit.safe_step(audit.stage_inputs)


Drive 체크포인트 복원 중...
base: 준비됨 — /content/drive/MyDrive/MMDL_data/mmmu_train_20261016/mmmu_train_20261016/package
expansion: 준비됨 — /content/drive/MyDrive/MMDL_data/mmmu_expansion_mmk12_20261115/mmmu_expansion_mmk12_20261115
중간 결과 Drive 저장 완료
중간 결과 Drive 저장 완료


{'base': PosixPath('/content/drive/MyDrive/MMDL_data/mmmu_train_20261016/mmmu_train_20261016/package'),
 'expansion': PosixPath('/content/drive/MyDrive/MMDL_data/mmmu_expansion_mmk12_20261115/mmmu_expansion_mmk12_20261115')}

## 4. E5 — 전체 문항과 이미지 점검

다음 두 셀을 순서대로 실행합니다. 원본에 여러 언어가 포함되어 있으므로 언어를 자동 번역하지 않습니다. 문자 종류 통계의 ‘한자’는 중국어·일본어 등을 엄밀히 구분한 언어 판정이 아닙니다.

풀이 보유량은 `reasoning`, `rationale`, `solution`을 확인합니다. **풀이가 있다는 것만 확인하며 내용이 맞는지는 E8에서 검사해야 합니다.** 길이는 문자 수이며 Qwen의 실제 토큰 수가 아닙니다.


In [ ]:
#@title 4-1. E5 문항·풀이·상태 충돌·출처 검사
try:
    audit.safe_step(audit.e5_import)
except Exception:
    audit.export_reports()  # 가능한 범위의 오류/부분 보고서를 남깁니다.
    raise


중간 결과 Drive 저장 완료
중간 결과 Drive 저장 완료


In [ ]:
#@title 4-2. E5 이미지 존재·손상·해시·연결 검사
try:
    audit.safe_step(audit.e5_images)
finally:
    E5_SUMMARY = audit.export_reports()
print("문항 수:", E5_SUMMARY["문항수"])
print("풀이 보유량:", E5_SUMMARY["상태별"].get("풀이_있음", 0))
print("E5 확인사항:", E5_SUMMARY["문제유형별건수"])
print("이 단계에서 E6 미완료 표시는 정상입니다.")


E5 이미지 검사: 남은 0개
중간 결과 Drive 저장 완료
중간 결과 Drive 저장 완료
중간 결과 Drive 저장 완료

보고서 저장: /content/drive/MyDrive/MMDL_audit_results/e5e6_20261116_v1/MMDL_E5_E6_결과.zip
문항 56,860개 / 검토 필요 56,860개 / E6 완료 False
문항 수: 56860
풀이 보유량: 18167
E5 확인사항: {'객관식_정답이_글자_아님': 552, '학습_정답과_원본_정답_다름': 115, '상태충돌_training_admitted': 9580, '상태충돌_difficulty_verified': 9434, '텍스트_후보_상한_도달': 206}
이 단계에서 E6 미완료 표시는 정상입니다.


## 5. E6 — 비교할 평가셋 준비

MMMU dev 150 / validation 900 / test 10,500문항 및 MMMU-Pro의 3가지 구성을 읽습니다. Pro의 각 구성은 1,730문항인지 확인합니다. 다운로드 시 네트워크·저장 공간이 추가로 필요합니다.

- 평가셋은 **문제·선택지·이미지 비교용**입니다. 정답·해설 열은 점검 DB에 가져오지 않습니다.
- 데이터 수·구성이 기대와 다르거나 평가 이미지 읽기에 문제가 생기면 E6를 완료 처리하지 않습니다.
- 저장한 `벤치마크_범위.json`의 커밋은 최종 평가 담당에게 함께 전달하세요.


In [ ]:
#@title 5-1. MMMU / MMMU-Pro 다운로드와 범위 검증
try:
    audit.safe_step(audit.load_benchmarks)
finally:
    audit.export_reports()


중간 결과 Drive 저장 완료
평가 데이터 다운로드: MMMU/MMMU @ 98e6ac0cb9b7b2cd2c991b85a50762edc4aedc68


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 92 files:   0%|          | 0/92 [00:00<?, ?it/s]

Agriculture/dev-00000-of-00001.parquet:   0%|          | 0.00/22.1M [00:00<?, ?B/s]

Accounting/dev-00000-of-00001.parquet:   0%|          | 0.00/273k [00:00<?, ?B/s]

Accounting/validation-00000-of-00001.par(…):   0%|          | 0.00/1.54M [00:00<?, ?B/s]

... (96 outputs left out)

vision/test-00001-of-00004.parquet:   0%|          | 0.00/469M [00:00<?, ?B/s]

vision/test-00002-of-00004.parquet:   0%|          | 0.00/318M [00:00<?, ?B/s]

vision/test-00003-of-00004.parquet:   0%|          | 0.00/446M [00:00<?, ?B/s]

중간 결과 Drive 저장 완료
  standard (10 options)/test-00000-of-00002.parquet: 865문항
  standard (10 options)/test-00001-of-00002.parquet: 865문항
  standard (4 options)/test-00000-of-00002.parquet: 865문항
  standard (4 options)/test-00001-of-00002.parquet: 865문항
  vision/test-00000-of-00004.parquet: 433문항
  vision/test-00001-of-00004.parquet: 433문항
  vision/test-00002-of-00004.parquet: 432문항
  vision/test-00003-of-00004.parquet: 432문항
중간 결과 Drive 저장 완료
중간 결과 Drive 저장 완료
중간 결과 Drive 저장 완료

보고서 저장: /content/drive/MyDrive/MMDL_audit_results/e5e6_20261116_v1/MMDL_E5_E6_결과.zip
문항 56,860개 / 검토 필요 56,860개 / E6 완료 False


## 6. E6 — 텍스트와 이미지 중복 후보 검사

정규화 텍스트 일치, 선택지 순서를 무시한 일치, MinHash 검색 후보의 Jaccard 유사도, 이미지 파일·픽셀 일치, pHash를 검사합니다. 이미지가 같아도 다른 문제일 수 있으므로 자동 삭제하지 않습니다.

텍스트 LSH는 근사 후보 검색이며 일부 누락 가능성이 있습니다. 후보 상한에 도달한 문항은 보고서에 따로 표시합니다. 학습 간 동일 문항 후보는 두 입력 묶음 내부와 묶음 사이 모두 검사합니다.


In [ ]:
#@title 6-1. E6 텍스트 검사
try:
    audit.safe_step(audit.e6_text)
finally:
    audit.export_reports()


E6 텍스트 검사가 이미 완료되었습니다.
중간 결과 Drive 저장 완료
중간 결과 Drive 저장 완료

보고서 저장: /content/drive/MyDrive/MMDL_audit_results/e5e6_20261116_v1/MMDL_E5_E6_결과.zip
문항 56,860개 / 검토 필요 56,860개 / E6 완료 False


In [ ]:
#@title 6-2. E6 이미지 SHA256 / 픽셀 / pHash 검사
try:
    audit.safe_step(audit.e6_images)
finally:
    audit.export_reports()


E6 이미지 해시 검사가 이미 완료되었습니다.
중간 결과 Drive 저장 완료
중간 결과 Drive 저장 완료

보고서 저장: /content/drive/MyDrive/MMDL_audit_results/e5e6_20261116_v1/MMDL_E5_E6_결과.zip
문항 56,860개 / 검토 필요 56,860개 / E6 완료 False


## 7. E6 — A100으로 CLIP 이미지 유사도 검색

학습 이미지와 평가셋 이미지를 비교합니다. CLIP은 그림 유형이 비슷한 서로 다른 문제도 높게 평가할 수 있으므로 **0.95 이상은 검토 후보**입니다. 이미지별 상위 5개를 저장하고 추가 후보가 있으면 표시합니다. 학습 이미지끼리의 CLIP 검색은 하지 않습니다(학습 간 pHash·해시는 앞 단계에 포함).

GPU가 없으면 여기서 멈춥니다. CPU 점검만 원하면 처음부터 `CLIP_ENABLED=False`와 새 `RUN_NAME`으로 실행하세요. GPU 메모리 부족으로 실패하면 배치 크기를 줄여 새 이름으로 실행할 수 있습니다. 정상 A100에서는 먼저 기본 배치를 사용하세요.


In [ ]:

#@title 7-1. CLIP 이미지 특징 계산 및 평가셋 비교
try:
    audit.safe_step(audit.e6_clip)
finally:
    audit.export_reports()


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIP 이미지 특징 계산: 남은 27,104개
  CLIP 64/27,104
중간 결과 Drive 저장 완료
  CLIP 128/27,104
  CLIP 192/27,104
중간 결과 Drive 저장 완료
  CLIP 256/27,104
  CLIP 320/27,104
중간 결과 Drive 저장 완료
  CLIP 384/27,104
  CLIP 448/27,104
중간 결과 Drive 저장 완료
  CLIP 512/27,104
  CLIP 576/27,104
중간 결과 Drive 저장 완료
  CLIP 640/27,104
  CLIP 704/27,104
중간 결과 Drive 저장 완료
  CLIP 768/27,104
  CLIP 832/27,104
중간 결과 Drive 저장 완료
  CLIP 896/27,104
  CLIP 960/27,104
중간 결과 Drive 저장 완료
  CLIP 1,024/27,104
  CLIP 1,088/27,104
중간 결과 Drive 저장 완료
  CLIP 1,152/27,104
  CLIP 1,216/27,104
중간 결과 Drive 저장 완료
  CLIP 1,280/27,104
  CLIP 1,344/27,104
중간 결과 Drive 저장 완료
  CLIP 1,408/27,104
  CLIP 1,472/27,104
중간 결과 Drive 저장 완료
  CLIP 1,536/27,104
  CLIP 1,600/27,104
중간 결과 Drive 저장 완료
  CLIP 1,664/27,104
  CLIP 1,728/27,104
중간 결과 Drive 저장 완료
  CLIP 1,792/27,104
  CLIP 1,856/27,104
중간 결과 Drive 저장 완료
  CLIP 1,920/27,104
  CLIP 1,984/27,104
중간 결과 Drive 저장 완료
  CLIP 2,048/27,104
  CLIP 2,112/27,104
중간 결과 Drive 저장 완료
  CLIP 2,176/27,104
  CLIP 2,240/27,10

## 8. 결과 확인 및 공유

가장 먼저 `결과요약.md`, `E5_출처별.csv`, `검토필요_문항.csv`, `E6_중복의심_쌍.csv`를 확인하세요.

| 결과 파일 | 쓰임 |
|---|---|
| `E5_문항별_결과.csv` | 각 문항의 풀이·출처·E5/E6 상태 |
| `E5_문제목록.csv` | 누락·정답 불일치·이미지 오류·학습 승인 상태 충돌 |
| `E5_과목별.csv`, `E5_출처별.csv` | 전체 구성과 출처별 풀이 보유량 |
| `E6_중복의심_쌍.csv` | 학습 문항과 평가 문항/다른 학습 문항의 후보 연결 |
| `E6_학습내외_중복그룹.csv` | 같은 텍스트·이미지 그룹 |
| `E6_기존기록과_이번검사.jsonl` | 기존 제공자의 기록과 이번 직접 검사 결과를 분리 |
| `E8_검토후보_ID.jsonl` | 풀이 검증을 진행할 후보 ID. **학습용 데이터 아님** |

**`MMDL_E5_E6_결과.zip`을 이 채팅에 보내면 다음 정제·풀이 검증 대상을 정할 수 있습니다.** ZIP에는 보고서만 들어가고 원본 이미지나 대형 체크포인트는 들어가지 않습니다. 원문 문제 일부는 검토용 표에 포함됩니다.


In [ ]:
#@title 8-1. 최종 결과 ZIP 생성
SUMMARY = audit.export_reports()
print(json.dumps({k:v for k,v in SUMMARY.items() if k not in {"벤치마크_범위"}}, ensure_ascii=False, indent=2))
RESULT_ZIP = OUTPUT_DIR / "MMDL_E5_E6_결과.zip"
print("공유할 파일:", RESULT_ZIP)


In [ ]:
#@title 8-2. 결과 ZIP을 PC로 다운로드 (원할 때 실행)
from google.colab import files
files.download(str(RESULT_ZIP))


In [ ]:
# 1. 완료한 실행 불러오기
%pip -q install pyahocorasick zstandard
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from contextlib import ExitStack
import hashlib, json, re, shutil, sqlite3, tempfile, unicodedata, tarfile, zipfile
import pandas as pd
import ahocorasick, zstandard
from IPython.display import display
from PIL import Image
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
RUN = Path(globals().get('OUTPUT_DIR','/content/drive/MyDrive/MMDL_audit_results/e5e6_20261116_v1'))
SHARED = RUN/'추가검증'  #
cfg=json.loads((RUN/'실행설정.json').read_text())
summary=json.loads((RUN/'reports/요약.json').read_text())
assert summary['E5_단계완료'] and summary['E6_설정된검사완료'], '완료한 실행 폴더를 지정하세요.'
local=Path(tempfile.mkdtemp())
shutil.copyfile(RUN/'audit.sqlite',local/'audit.sqlite')
db=sqlite3.connect((local/'audit.sqlite').as_uri()+'?mode=ro',uri=True)
items={rid:dict(json.loads(data),pkg=pkg,id=i,rid=rid) for rid,pkg,i,data in db.execute('SELECT * FROM examples ORDER BY rid')}
assert len(items)==summary['문항수']
def fingerprint(x): return hashlib.sha256(str(x).encode()).hexdigest()
status=pd.read_csv(RUN/'reports/E5_문항별_결과.csv',dtype=str,keep_default_na=False).set_index('행키',verify_integrity=True)
assert set(status.index)==set(items)
def src(d): return str(d.get('source_dataset') or d['source_family'])
def mmmu(d):
    return any(re.search(r'mmmu',str(d.get(k,'')).casefold()) for k in ('source_dataset','source_family','id'))
def csvfile(frame,path):
    tmp=path.with_suffix('.tmp'); frame.to_csv(tmp,index=False,encoding='utf-8-sig');tmp.replace(path)
# 동일 자료의 검토표만 재사용합니다.
stamp=hashlib.sha256()
for d in items.values(): stamp.update(json.dumps(d,sort_keys=True,ensure_ascii=False).encode())
work=SHARED/(RUN.name+'_'+stamp.hexdigest()[:12]); work.mkdir(parents=True,exist_ok=True)
sources=defaultdict(list)
for rid,d in items.items(): sources[src(d)].append(rid)
samples={s:sorted(ids,key=lambda i:fingerprint('42/'+i))[:20] for s,ids in sources.items()}
print('실제 후보 출처',len(sources),'개 / 문항',len(items),'개 / 결과',work)

실제 후보 출처 43 개 / 문항 56860 개 / 결과 /content/drive/MyDrive/MMDL_audit_results/e5e6_20261116_v1/추가검증/e5e6_20261116_v1_8ffbfc0605c1


In [ ]:
# 2. 표본과 검토 기록표. 기존 작성 내용은 보존합니다.
review_path=work/'E5_출처검토_작성.csv'
sample_rows=[]; overview=[]
for s,ids in sorted(sources.items()):
    selected=samples[s]
    for rid in selected:
        d=items[rid]
        sample_rows.append(dict(데이터셋=s,행키=rid,원본ID=d['id'],질문=d['question'],선택지=d['options'],
                                정답=d['answer'],풀이=d['solutions'],이미지=d['images']))
    overview.append(dict(데이터셋=s,전체=len(ids),표본수=len(selected),
        객관식수=sum(bool(items[i]['options']) for i in ids),풀이보유수=sum(bool(items[i]['solutions']) for i in ids),
        기록라이선스=' | '.join(sorted({str(items[i]['source_license']) for i in ids})),
        형식='',정답검증가능='',풀이포함='',이미지필요='',라이선스확인='',MMMU출처='',이상한행키='',검토자='',결정='대기'))
sample_df=pd.DataFrame(sample_rows)
csvfile(sample_df,work/'E5_20건표본.csv')
if not review_path.exists(): csvfile(pd.DataFrame(overview),review_path)
display(pd.DataFrame(overview)[['데이터셋','전체','표본수','객관식수','풀이보유수','기록라이선스']])
print(review_path)
# 발췌한 20건 전부 보기 (20건 미만인 출처는 전수)
def 표본보기(데이터셋):
    with pd.option_context('display.max_rows',20,'display.max_colwidth',None):
        display(sample_df[sample_df['데이터셋']==데이터셋])
# 예: 표본보기(overview[0]['데이터셋'])

,데이터셋,전체,표본수,객관식수,풀이보유수,기록라이선스
0,ArtQuest,1,1,0,0,CC-BY-4.0 (ArtQuest annotation dataset); SemAr...
1,ChongyanChen/VQAonline,48,20,0,0,CC-BY-SA-4.0
2,ClareNie/PersonaVLM-Dataset,106,20,0,0,apache-2.0
3,FanqingM/MMK12,9434,20,0,0,Apache-2.0 (publisher LICENSE)
4,HuggingFaceM4/FineVision,870,20,120,0,CC-BY-NC-SA-4.0 | GPL-3.0 (source terms apply)...
5,JulianAT/InteriorlyQA,1,1,0,0,MIT
6,MING-ZCH/TFQ-Data-Full,31,20,0,0,CC-BY-NC-4.0
7,MMInstruction/ArxivQA,9878,20,9878,9867,CC-BY-SA-4.0 (OpenAI terms apply)
8,NExTplusplus/TAT-DQA,24,20,0,0,CC-BY-4.0
9,NExTplusplus/TAT-HQA,91,20,0,0,Apache-2.0 (TAT-HQA repository); inherited TAT...


/content/drive/MyDrive/MMDL_audit_results/e5e6_20261116_v1/추가검증/e5e6_20261116_v1_8ffbfc0605c1/E5_출처검토_작성.csv


In [ ]:
# 2-1. 표본 이미지 확인 및 내보내기에 공통으로 쓰는 파일 복사 함수
INPUT_PATHS=dict(cfg['inputs'])
def copy_images(rids,dest):
    tasks=defaultdict(dict)
    for rid in rids:
        d=items[rid]
        for rel in d['images']:
            if Path(rel).is_absolute() or '..' in Path(rel).parts: raise ValueError(rel)
            tasks[d['pkg']][rel]=dest/d['pkg']/rel
    for pkg,paths in tasks.items():
        origin=Path(INPUT_PATHS[pkg])
        active=globals().get('audit')
        root=getattr(active,'roots',{}).get(pkg) if getattr(active,'output',None)==RUN else None
        if not root and origin.is_dir():
            found=[p.parent for p in origin.rglob('train_messages.jsonl') if (p.parent/'reservoir_candidates.jsonl').exists()]
            if len(found)!=1: raise ValueError('원본 폴더 확인 '+pkg)
            root=found[0]
        def save(rel,stream):
            path=paths[rel]; path.parent.mkdir(parents=True,exist_ok=True)
            with stream as f,path.open('wb') as w: shutil.copyfileobj(f,w)
            stored,=db.execute('SELECT data FROM images WHERE ikey=?',(pkg+'/'+rel,)).fetchone()
            expected=json.loads(stored or '{}').get('sha256')
            if expected and hashlib.sha256(path.read_bytes()).hexdigest()!=expected: raise ValueError('이미지 변경 '+rel)
        if root:
            for rel in paths: save(rel,(Path(root)/rel).open('rb'))
            continue
        pending=set(paths)
        def match(name):
            parts=Path(name).parts
            return next((k for i in range(len(parts)) if (k:='/'.join(parts[i:])) in pending),None)
        with ExitStack() as stack:
            if origin.suffix=='.zip':
                arc=stack.enter_context(zipfile.ZipFile(origin))
                for name in arc.namelist():
                    rel=match(name)
                    if rel: save(rel,arc.open(name)); pending.remove(rel)
                    if not pending: break
            else:
                raw=stack.enter_context(origin.open('rb'))
                if origin.suffix=='.zst': raw=stack.enter_context(zstandard.ZstdDecompressor().stream_reader(raw))
                arc=stack.enter_context(tarfile.open(fileobj=raw,mode='r|*'))
                for member in arc:
                    rel=match(member.name)
                    if rel and member.isfile(): save(rel,arc.extractfile(member)); pending.remove(rel)
                    arc.members.clear()
                    if not pending: break
        if pending: raise FileNotFoundError(str(sorted(pending)[:3]))
def 표본이미지보기(데이터셋):
    ids=samples[데이터셋]
    for rid in ids:
        d=items[rid]
        print(rid,d['question'],d['options'],'정답',d['answer'],'풀이',d['solutions'])
        for rel in d['images']:
            try:
                path=local/'sample'/d['pkg']/rel
                with Image.open(path) as im: im.thumbnail((900,900)); display(im.copy())
            except OSError as exc: print('이미지 확인 필요',rel,exc)
# 손상/누락은 E5 표에 이미 표시됨. 정상으로 확인된 문항 이미지부터 복사합니다.
sample_ids=[i for ids in samples.values() for i in ids]
valid_sample=[i for i in sample_ids if all((lambda r:r and json.loads(r[0] or '{}').get('status')=='정상')(
    db.execute('SELECT data FROM images WHERE ikey=?',(items[i]['pkg']+'/'+p,)).fetchone()) for p in items[i]['images'])]
copy_images(valid_sample,local/'sample')
# 표본이미지보기(overview[0]['데이터셋'])

In [ ]:
# 3. 평가셋 질문과 정확 일치 + 양방향 부분 포함. GPU 불필요.
def normalize(text):
    s=unicodedata.normalize('NFKC',str(text or '')).casefold()
    return ' '.join(''.join(' ' if unicodedata.category(c)[0] in 'PS' else c for c in s).split())
bench={}
coverage=Counter(); empty=Counter()
for bid,repo,config,split,q in db.execute('SELECT bid,repo,config,split,q FROM bench ORDER BY bid'):
    key=(repo,split if repo=='MMMU/MMMU' else config)
    coverage[key]+=1
    q=normalize(q)
    if q: bench.setdefault(q,bid)
    else: empty[key]+=1
assert {s:n for (r,s),n in coverage.items() if r=='MMMU/MMMU'}=={'dev':150,'validation':900,'test':10500}
for c in ('standard (4 options)','standard (10 options)','vision'):
    assert sum(n for (r,k),n in coverage.items() if r=='MMMU/MMMU_Pro' and k.split('/')[-1]==c)==cfg['config'].get('expected_pro_rows_per_config',1730)
coverage_df=pd.DataFrame([{'평가셋':r,'범위':k,'전체':v,'텍스트비교':v-empty[r,k],'빈질문':empty[r,k]} for (r,k),v in coverage.items()])
display(coverage_df); csvfile(coverage_df,work/'E6_평가범위.csv')
def containment(questions,refs):
    # 각 방향에서 하나의 증거만 필요. 동일 문항이 여러 평가 문항에 걸려도 1건.
    def index(words):
        a=ahocorasick.Automaton()
        for word in words: a.add_word(word,word)
        if len(a): a.make_automaton()
        return a
    match={q:('완전 일치',refs[q],'동일') for q in questions if q in refs}
    a=index(refs)
    for q in questions:
        if not q or q in match or not len(a): continue
        b=next((v for _,v in a.iter(q)),None)
        if b: match[q]=('부분 포함',refs[b],'평가 질문이 학습 질문 안에 포함')
    rev=index(q for q in questions if q and q not in match)
    if len(rev):
        for b,bid in refs.items():
            for _,q in rev.iter(b): match.setdefault(q,('부분 포함',bid,'학습 질문이 평가 질문 안에 포함'))
    return match
questions={rid:normalize(d['question']) for rid,d in items.items()}
matches=containment(set(questions.values()),bench)
print('질문이 같은 원본 행도 개별 행키로 추적합니다. 출처별 고유 문항 집계는 다음 셀에 표시됩니다.')

,평가셋,범위,전체,텍스트비교,빈질문
0,MMMU/MMMU_Pro,standard (10 options),1730,1730,0
1,MMMU/MMMU_Pro,standard (4 options),1730,1730,0
2,MMMU/MMMU_Pro,vision,1730,0,1730
3,MMMU/MMMU,dev,150,150,0
4,MMMU/MMMU,test,10500,10500,0
5,MMMU/MMMU,validation,900,900,0


질문이 같은 원본 행도 개별 행키로 추적합니다. 출처별 고유 문항 집계는 다음 셀에 표시됩니다.


In [ ]:
# 4. 가이드 충족 현황 + 데이터셋별 제거 건수 (검토 CSV 수정 후 재실행)
review=pd.read_csv(review_path,keep_default_na=False,dtype=str).set_index('데이터셋',verify_integrity=True)
assert set(review.index)==set(sources),'검토표의 출처 목록을 확인하세요.'
allowed=set(); skip=set(); finished=set(); source_reasons={}
for s,r in review.iterrows():
    ids={x.strip() for x in r['이상한행키'].split(',') if x.strip()}
    if not ids.issubset(set(sources[s])): raise ValueError('이상한행키 확인 '+s)
    skip.update(ids)
    reviewed=all(r[k].strip() for k in ('형식','정답검증가능','풀이포함','이미지필요','라이선스확인','검토자')) and r['MMMU출처'] in ('예','아니오','불명')
    if reviewed and (r['결정']=='제외' or (r['결정']=='통과' and r['MMMU출처']=='아니오')): finished.add(s)
    if reviewed and r['결정']=='통과' and r['MMMU출처']=='아니오': allowed.add(s)
    source_reasons[s]='출처 제외' if r['결정']=='제외' or r['MMMU출처']=='예' else '출처 검토 대기'
log=[]; ready=[]
id_counts=Counter((src(d),d['id']) for d in items.values())
for rid,d in items.items():
    st=status.loc[rid]; s=src(d); q=questions[rid]
    kind,bid,direction=matches.get(q,('','',''))
    if mmmu(d): outcome='MMMU 계열 출처 제외'
    elif s not in allowed: outcome=source_reasons[s]
    elif id_counts[s,d['id']]>1 or rid in skip or st['E5상태']!='기계검사_이상없음' or not q: outcome='E5 보류·제외'
    elif kind: outcome=kind
    elif st['E6상태']!='구현범위에서_미검출' or st['학습간의심종류']: outcome='기존 E6 의심·검사제한 보류'
    else: outcome='통과'; ready.append(rid)
    log.append(dict(데이터셋=s,행키=rid,원본ID=d['id'],텍스트검출=kind,평가ID예시=bid,포함방향=direction,결과=outcome))
ledger=pd.DataFrame(log)
csvfile(ledger,work/'문항별_판정과근거.csv')
eligible=['완전 일치','부분 포함','기존 E6 의심·검사제한 보류','통과']
table=[]
for s,group in ledger.groupby('데이터셋'):
    n=Counter(group['결과']); total=sum(n[k] for k in eligible)
    table.append(dict(데이터셋=s,원본행=len(group),출처및E5보류제외=len(group)-total,전체=total,
        완전일치=n['완전 일치'],부분포함=n['부분 포함'],제거후=total-n['완전 일치']-n['부분 포함'],
        추가보류=n[eligible[2]],인계가능=n['통과']))
counts=pd.DataFrame(table); display(counts); csvfile(counts,work/'E6_출처별_제거건수.csv')
def dashboard(exported=False):
    rows=[
        ('E5','검사한 데이터셋 목록','충족',f'{len(sources)}개 / E5_출처검토_작성.csv'),
        ('E5','출처별 20건 표본 추출','충족',f'{len(sample_df)}건 (20건 미만 전수)'),
        ('E5','표본 열람·형식·정답 검증 가능성·풀이·이미지 필요·라이선스·이상 ID 기록','충족' if len(finished)==len(sources) else '사람 검토 필요',f'{len(finished)}/{len(sources)}개 출처 기록'),
        ('E5','MMMU·Pro 원출처 제외','충족' if len(finished)==len(sources) else '부분 충족','출처명·ID 표시 검사 및 출처 미확인 보류'),
        ('E6','MMMU dev/validation/test 및 Pro 전체 범위 확인','충족','E6_평가범위.csv / 이미지 전용 빈 질문은 텍스트 비교 불가'),
        ('E6','소문자·공백·기호 정규화, 완전 일치, 양방향 부분 포함','충족',f'비어 있지 않은 평가 질문 {sum(coverage.values())-sum(empty.values())}행'),
        ('E6','출처별 전체·완전 일치·부분 포함·제거 후 집계','충족' if len(finished)==len(sources) else '잠정',f'현재 E5 통과분에서 제거 {int(counts.완전일치.sum()+counts.부분포함.sum())}건'),
        ('E5/E6','공통 JSONL·이미지·mmmu_overlap=false 인계','충족' if exported and len(finished)==len(sources) else ('부분 인계' if exported else '미출력'),f'현재 인계 가능 {len(ready)}건'),
        ('실행','추가 분석 GPU 불필요','충족','CPU 실행')]
    rows.extend([('E5','기존 형식·정답 필드·이미지 점검','충족',f'검사 {len(status)}행, 기계검사 이상없음 {(status["E5상태"]=="기계검사_이상없음").sum()}행'),
                 ('E6','기존 해시·유사도 의심 문항 반영','충족','기존 E6 의심·검사제한·학습간 중복을 인계에서 보류')])
    df=pd.DataFrame(rows,columns=['단계','요구사항','상태','근거'])
    df.insert(2,'기존 코드 기준',['충족','미충족','미충족','부분 충족','충족','부분 충족','미충족','미충족','CPU 가능','충족','충족'])
    display(df);csvfile(df,work/'요구사항_충족현황.csv')
dashboard()
print('전체는 출처·E5 통과 후 텍스트 검사를 집계한 문항 수입니다. 검토 전에는 제거 수가 잠정 0일 수 있습니다.')

,데이터셋,원본행,출처및E5보류제외,전체,완전일치,부분포함,제거후,추가보류,인계가능
0,ArtQuest,1,1,0,0,0,0,0,0
1,ChongyanChen/VQAonline,48,48,0,0,0,0,0,0
2,ClareNie/PersonaVLM-Dataset,106,106,0,0,0,0,0,0
3,FanqingM/MMK12,9434,9434,0,0,0,0,0,0
4,HuggingFaceM4/FineVision,870,870,0,0,0,0,0,0
5,JulianAT/InteriorlyQA,1,1,0,0,0,0,0,0
6,MING-ZCH/TFQ-Data-Full,31,0,31,0,0,31,0,31
7,MMInstruction/ArxivQA,9878,4,9874,0,0,9874,1838,8036
8,NExTplusplus/TAT-DQA,24,24,0,0,0,0,0,0
9,NExTplusplus/TAT-HQA,91,91,0,0,0,0,0,0


,단계,요구사항,기존 코드 기준,상태,근거
0,E5,검사한 데이터셋 목록,충족,충족,43개 / E5_출처검토_작성.csv
1,E5,출처별 20건 표본 추출,미충족,충족,624건 (20건 미만 전수)
2,E5,표본 열람·형식·정답 검증 가능성·풀이·이미지 필요·라이선스·이상 ID 기록,미충족,사람 검토 필요,29/43개 출처 기록
3,E5,MMMU·Pro 원출처 제외,부분 충족,부분 충족,출처명·ID 표시 검사 및 출처 미확인 보류
4,E6,MMMU dev/validation/test 및 Pro 전체 범위 확인,충족,충족,E6_평가범위.csv / 이미지 전용 빈 질문은 텍스트 비교 불가
5,E6,"소문자·공백·기호 정규화, 완전 일치, 양방향 부분 포함",부분 충족,충족,비어 있지 않은 평가 질문 15010행
6,E6,출처별 전체·완전 일치·부분 포함·제거 후 집계,미충족,잠정,현재 E5 통과분에서 제거 8건
7,E5/E6,공통 JSONL·이미지·mmmu_overlap=false 인계,미충족,미출력,현재 인계 가능 21293건
8,실행,추가 분석 GPU 불필요,CPU 가능,충족,CPU 실행
9,E5,기존 형식·정답 필드·이미지 점검,충족,충족,"검사 56860행, 기계검사 이상없음 46613행"


전체는 출처·E5 통과 후 텍스트 검사를 집계한 문항 수입니다. 검토 전에는 제거 수가 잠정 0일 수 있습니다.


In [ ]:
# 5. 통과한 자료와 이미지를 새 결과 폴더에 저장 (이전 결과 보존)
if not ready:
    print('인계 가능 문항 0건. 요구사항_충족현황.csv와 문항별_판정과근거.csv를 확인하세요.')
else:
    tag=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%f')
    staging=work/('작성중_'+tag); data=staging/'data'; data.mkdir(parents=True)
    copy_images(ready,data/'images')
    with ExitStack() as stack:
        handles={}; seen=set()
        for rid in ready:
            d=items[rid]; s=src(d)
            if s not in handles:
                name=re.sub(r'[^\w.-]+','_',s)[:75]+'_'+fingerprint(s)[:8]
                handles[s]=stack.enter_context((data/f'train_{name}.jsonl').open('w',encoding='utf8'))
            ident=d['pkg']+'/'+d['id']
            assert ident not in seen; seen.add(ident)
            record=dict(id=ident,original_id=d['id'],rid=rid,source=s,images=['images/'+d['pkg']+'/'+p for p in d['images']],
                        question=d['question'],options=d['options'],answer=str(d['answer']).strip(),
                        solution=next(iter(d['solutions'].values()),None),mmmu_overlap=False,license=d['source_license'])
            handles[s].write(json.dumps(record,ensure_ascii=False)+'\n')
    # E8 입력은 기존 후보와 이번 인계 문항의 교집합으로 제한합니다.
    e8=staging/'E8_검토후보_ID.jsonl'; accepted=set(ready)
    with (RUN/'reports'/e8.name).open() as f,e8.open('w',encoding='utf8') as w:
        for line in f:
            if json.loads(line)['rid'] in accepted: w.write(line.rstrip()+'\n')
    for p in (review_path,work/'E6_출처별_제거건수.csv',work/'문항별_판정과근거.csv',work/'E6_평가범위.csv'):
        shutil.copyfile(p,staging/p.name)
    final=work/('결과_'+tag); staging.rename(final);dashboard(True)
    shutil.copyfile(work/'요구사항_충족현황.csv',final/'요구사항_충족현황.csv')
    print('공유할 폴더',final,'/ 문항',len(ready),'개')
    print('기존 E8 노트북에서 candidate_file 줄을 아래 경로로 바꾸세요.')
    print('candidate_file = Path('+repr(str(final/e8.name))+')')
    print("그 바로 아래 후보수 assert를 다음으로 바꾸세요: assert len(cs) <= s['E8_후보문항수'] and len({c['rid'] for c in cs}) == len(cs)")

,단계,요구사항,기존 코드 기준,상태,근거
0,E5,검사한 데이터셋 목록,충족,충족,43개 / E5_출처검토_작성.csv
1,E5,출처별 20건 표본 추출,미충족,충족,624건 (20건 미만 전수)
2,E5,표본 열람·형식·정답 검증 가능성·풀이·이미지 필요·라이선스·이상 ID 기록,미충족,사람 검토 필요,29/43개 출처 기록
3,E5,MMMU·Pro 원출처 제외,부분 충족,부분 충족,출처명·ID 표시 검사 및 출처 미확인 보류
4,E6,MMMU dev/validation/test 및 Pro 전체 범위 확인,충족,충족,E6_평가범위.csv / 이미지 전용 빈 질문은 텍스트 비교 불가
5,E6,"소문자·공백·기호 정규화, 완전 일치, 양방향 부분 포함",부분 충족,충족,비어 있지 않은 평가 질문 15010행
6,E6,출처별 전체·완전 일치·부분 포함·제거 후 집계,미충족,잠정,현재 E5 통과분에서 제거 8건
7,E5/E6,공통 JSONL·이미지·mmmu_overlap=false 인계,미충족,부분 인계,현재 인계 가능 21293건
8,실행,추가 분석 GPU 불필요,CPU 가능,충족,CPU 실행
9,E5,기존 형식·정답 필드·이미지 점검,충족,충족,"검사 56860행, 기계검사 이상없음 46613행"


공유할 폴더 /content/drive/MyDrive/MMDL_audit_results/e5e6_20261116_v1/추가검증/e5e6_20261116_v1_8ffbfc0605c1/결과_20261119T140534_783794 / 문항 21293 개
기존 E8 노트북에서 candidate_file 줄을 아래 경로로 바꾸세요.
candidate_file = Path('/content/drive/MyDrive/MMDL_audit_results/e5e6_20261116_v1/추가검증/e5e6_20261116_v1_8ffbfc0605c1/결과_20261119T140534_783794/E8_검토후보_ID.jsonl')
그 바로 아래 후보수 assert를 다음으로 바꾸세요: assert len(cs) <= s['E8_후보문항수'] and len({c['rid'] for c in cs}) == len(cs)


In [ ]:
print("① 문항이 인계되지 않은 이유와 건수")
display(ledger["결과"].value_counts().rename("문항 수").to_frame())

print("② 출처별 검토 결과 — 4번 셀이 마지막으로 읽은 값")
display(review[["결정", "MMMU출처", "라이선스확인", "검토자"]])

print("③ 수정해야 하는 검토표의 정확한 위치")
print(review_path)

① 문항이 인계되지 않은 이유와 건수


,문항 수
결과,
통과,21293
기존 E6 의심·검사제한 보류,15389
출처 검토 대기,10246
E5 보류·제외,9438
출처 제외,486
부분 포함,8


② 출처별 검토 결과 — 4번 셀이 마지막으로 읽은 값


,결정,MMMU출처,라이선스확인,검토자
데이터셋,,,,
ArtQuest,대기,아니오,Zenodo의 QA 배포 확인. 이 패키지에 쓰인 작품 이미지 이용 조건 미확인,ChatGPT (AI 표본 검토)
ChongyanChen/VQAonline,제외,아니오,CC BY-SA 4.0 (배포 카드),ChatGPT (AI 표본 검토)
ClareNie/PersonaVLM-Dataset,제외,아니오,Apache-2.0 (배포 카드),ChatGPT (AI 표본 검토)
FanqingM/MMK12,통과,아니오,Apache-2.0 (배포본 manifest와 배포자 LICENSE),ChatGPT (AI 표본 검토)
HuggingFaceM4/FineVision,대기,불명,패키지 표기 ODC-BY-1.0. 구성 출처별 조건 추가 적용,ChatGPT (AI 표본 검토)
JulianAT/InteriorlyQA,제외,불명,원 배포본의 데이터 이용 조건 미확인,ChatGPT (AI 표본 검토)
MING-ZCH/TFQ-Data-Full,통과,아니오,"CC BY-NC 4.0 (배포 카드, 비상업 연구)",ChatGPT (AI 표본 검토)
MMInstruction/ArxivQA,통과,아니오,CC BY-SA 4.0. 카드에 OpenAI 조건도 명시. 패키지 연구용 범위,ChatGPT (AI 표본 검토)
NExTplusplus/TAT-DQA,대기,아니오,공식 README의 데이터 라이선스 CC BY 4.0,ChatGPT (AI 표본 검토)


③ 수정해야 하는 검토표의 정확한 위치
/content/drive/MyDrive/MMDL_audit_results/e5e6_20261116_v1/추가검증/e5e6_20261116_v1_8ffbfc0605c1/E5_출처검토_작성.csv


## 재개 및 결과 해석

- Drive의 `audit.sqlite`가 체크포인트입니다. 삭제하지 마세요. `reports`와 결과 ZIP은 다시 생성할 수 있습니다.
- 런타임이 끊겼다면 **같은 코드·원본·설정·RUN_NAME으로 위에서부터** 실행하세요. 기존 문항·이미지 검사와 계산된 CLIP 특징은 복원합니다. 압축 해제·평가 이미지 준비·텍스트 검색 인덱스 구축은 로컬 캐시가 없어지면 다시 수행할 수 있습니다.
- 텍스트/pHash 검색 도중 끊긴 경우 해당 검색 단계는 재구축하지만 저장된 후보를 중복해서 늘리지 않습니다. 모든 단계가 초 단위로 재개되는 것은 아닙니다.
- 데이터 또는 임계값을 바꾸면 새 RUN_NAME을 써야 이전 결과와 섞이지 않습니다.
- `E5 단계 완료`는 ‘검사 실행 완료’입니다. 모든 문항이 합격했다는 뜻이 아닙니다.
- `구현범위에서 미검출`은 확정된 무오염 판정이 아닙니다. 기존 오염 검사 기록, 후보 검토, 빠진 검사 범위를 함께 확인하세요.
- `mmmu_overlap`은 수동 확정 전 `null`로 유지됩니다. 파일이 자동으로 학습 승인 상태가 되지 않습니다.
- E5/E6 점검에서 다루는 답 일치는 **두 파일에 기록된 답이 같은지**입니다. 답 자체가 맞는지·풀이가 이미지에 근거하는지는 E8의 별도 작업입니다.

## 검증 범위와 참고

제작 환경에서 합성 자료로 폴더·tar.zst 입력, 잘못된 경로, 문항/이미지 오류, 정확·유사 중복, 평가셋 누락 감지, 체크포인트 복원, 설정 변경 차단을 테스트했습니다. 사용자 미리보기의 실제 6문항도 E5로 처리해 풀이 2건과 상태 충돌 3건을 확인했습니다. **전체 사용자 원본, 실제 Colab A100의 CLIP 추론, 전체 온라인 벤치마크 다운로드는 제작 환경에서 끝까지 실행하지 않았습니다.**

- [MMMU 데이터](https://huggingface.co/datasets/MMMU/MMMU)
- [MMMU-Pro 데이터](https://huggingface.co/datasets/MMMU/MMMU_Pro)
- [Hugging Face snapshot_download](https://huggingface.co/docs/huggingface_hub/package_reference/file_download#huggingface_hub.snapshot_download)
- [CLIP 모델 문서](https://huggingface.co/docs/transformers/v4.57.2/model_doc/clip)

이 노트북은 기존 GitHub 저장소를 변경하지 않습니다.
